# cdli2unicode: CDLI to unicode cuneiform data conversion



##Task: Convert the transliteration (Latin sign readings in syllable form) from the CDLI data repository to unicode cuneiform using the [Nuolena](https://github.com/tosaja/Nuolenna/blob/master/sign_list.txt) and [Akkademia](https://github.com/gaigutherz/Akkademia/blob/master/cuneiform_to_unicode_fixed.csv) sign lists.

##Outline
0. Data Description
1. CDLI Data Acquisition
1.1. Point to Your Directory for Downloading
1.2. Download CDLI
1.3 Load in a Pandas Data Frame
1.4. Use the Catalog to Select Transliterations
1.5. Create a Data Frame with the Selected Texts
1.6. Complete for All Texts in the Dataset: cdli_atf
2. Sign List Dictionary
2.1. Nuolenna Sign List
2.2. Akkademia Sign List
3. Punctuation
3.1. Punctuation removal (in sequence)
3.2. Allographs and Compund Grapheme Modifiers
4. Match Sign Readings to Unicode Cuneiform
5. Evaluation for Resulting Unmatched Sign Readings
5.1. Export CSV
5.2. Assessment of Unicode Conversion
6. ORACC Format Conversion: cdli_forms
6.1. Punctuation removal
6.2. Match 'Clean' Sign Readings to Unicode Cuneiform
6.3. Assessment of Unicode Conversion
7. Include Leiden Transcription readings in Updated Sign List Dictionary

-----------
#[FactGrid Cuneiform Project](https://database.factgrid.de/wiki/FactGrid:Cuneiform_Project)

UCB Data Science Discovery Student: Harrison Lirui Huang


##0 Data description

See the original notebook: https://github.com/niekveldhuis/compass/blob/master/2_3_Data_Acquisition_CDLI/2_3_Data_Acquisition_CDLI.ipynb

The [Cuneiform Digital Library Initiative](https://cdli.mpiwg-berlin.mpg.de/), created by [Bob Englund](https://cdli.ucla.edu/?q=robert-k-englund) (UCLA) in the early two thousands, is a central repository for meta-data, images, and transliterations of cuneiform objects (translations are offered only for a small minority of texts). Today more than 335,000 objects are listed in the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) catalog, with tens of thousands of photographs and line drawings. Each object in [CDLI](https://cdli.mpiwg-berlin.mpg.de/) receives a unique ID number (the so-called P-number), and these numbers are widely used today in print and in on-line projects. Initially, [CDLI](https://cdli.mpiwg-berlin.mpg.de/) focused primarily on administrative texts from the third millennium, and this is still the area of its greatest strength. Currently, approximately 121,000 texts are available in transliteration in [CDLI](https://cdli.mpiwg-berlin.mpg.de/). Part of this corpus was produced by the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) team at UCLA, others were contributed by partners or were imported from other projects such as [ETCSL](https://etcsl.orinst.ox.ac.uk/) (for Sumerian literary texts), [DCCLT](http://oracc.org/dcclt) (for lexical texts), or [BDTNS](http://bdtns.filol.csic.es/) (for Ur III administrative texts). The photographs on [CDLI](https://cdli.mpiwg-berlin.mpg.de/) were largely produced in cooperative projects with museums all over the world, where [CDLI](https://cdli.mpiwg-berlin.mpg.de/) staff or partners would go to scan an entire collection or major parts of a collection. These images are copyright of the museum where the object is held and there is no wholesale downloading of the entire image set.

The [CDLI](https://cdli.mpiwg-berlin.mpg.de/) project transformed the practice of Assyriology in multiple ways. The availability of large numbers of photographs made it possible to collate problematic passages in texts only published in transliteration and/or hand drawing. Issuing ID numbers made it easier to refer to a particular tablet unambiguously, while avoiding the confusion of obscure publication abbreviations and museum numbers. The spread of all this information on the web made it available to everyone with an internet connection.

For each cuneiform object the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) catalog provides information about where it was published (and by whom), in which collection it is kept, where it was excavated, to which period it belongs, what textual genre it represents, etc. In addition, an object may be represented by one or more images (photographs and/or hand drawings) and by a transliteration. Several of the fields in the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) catalog either use a restricted vocabulary (period, genre) or have been standardized (provenance, author's name, owner, museum number), greatly facilitating search.

The issue of standardization is much more difficult for linguistic data in transliteration. Here, Sumerian and Akkadian pose rather different challenges. For Sumerian, there are two main issues. First, Sumerologists tend to use different sets of conventions for representing Sumerian words in the Latin alphabet. The word for "to give" is read [**šum₂**](http://oracc.org/epsd2/o0039914) by some, but **sum** by others. Similarly, the word for "ox" is read either [**gud**](http://oracc.org/epsd2/o0028670) or **gu₄**. These readings (**šum₂** vs **sum** or **gud** vs. **gu₄**) represent the same word and render the same cuneiform sign - they simply differ in modern transliteration conventions. Variation in such conventions has grown recently by the introduction of a new set of readings by P. Attinger (Bern), which has received wide following, in particular in Germany. Such variation in sign readings is based on the one hand on differing interpretations of the data from [ancient sign lists](http://oracc.org/dcclt/signlists) (which provide transcriptions of Sumerian words in Akkadian) and on the other hand on the definition of what an ideal transliteration should achieve (whether it should represent the abstract lexeme, or rather its concrete pronunciation, or something in between). For the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) search engine, which is based on a FileMaker database, such variation presents a problem when searching for (Sumerian) words. The solution has been to strictly impose a set of [preferred sign readings](https://cdli.ucla.edu/methods/sign_reading.html), a policy that has been carried out with admirable consistency.

:::{admonition} Attinger readings
:class: tip, dropdown
For the readings introduced by P. Attinger, see the introduction to his [Glossaire sumérien–français](https://www.harrassowitz-verlag.de/isbn_9783447116169.ahtml) (2021).
:::

Second, Sumerologists today have no good standard for word segmentation. In the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) data set one may find the word [**ninda-i₃-de₂-a**](http://oracc.org/epsd2/o0036259) (a pastry) transliterated as **ninda-i₃-de₂-a**, **ninda i₃-de₂-a**, **ninda i₃ de₂-a**, **nig₂-i₃-de₂-a**, **nig₂ i₃ de₂-a**, etcetera (**nig₂** and **ninda** are two different words, written by the same sign and there is no full agreement which of these is to be used in this particular expression). None of these various renderings is necessarily "wrong," because we know fairly little about the formation and segmentation of Sumerian nouns. For computational approaches this variation poses an important challenge.

For Akkadian the variation in reading conventions plays a much smaller role; for most dialects of Akkadian (with the exception of Old Akkadian) scholars generally agree on transliteration conventions; word segmentation is hardly ever a problem. For search engines, however, Akkadian transliteration is much more difficult to deal with because the same word may be spelled in many different ways. Without lemmatization, there is no way a machine can tell that ***ša-ar-ru-um***, ***ša-ar-ri-im***, ***ša-ar-ra-am***, ***šar-ru***, ***šar-ri***, ***šar-ra***, **LUGAL**, and **MAN** all represent forms of the same word for "king" in syllabic and logographic writing. The rich morphology of Akkadian, with prefixes, suffixes, and infixes and various vowel patterns to be applied to different forms of a single verb further complicates this issue.

Since [CDLI](https://cdli.mpiwg-berlin.mpg.de/) does not offer lemmatization, searching for words on this site is much more popular (and more useful) for Sumerian than it is for Akkadian. Sumerian words usually include the root of the word (written logographically) with prefixes and/or suffixes attached. Although spelling variations exist (e.g. **dag-si**, **da-ag-si**, **da-ag-ši-um**, and **da-ag-zi-um**, all representing variants of the word [dagsi](http://oracc.org/epsd2/o0025593) for saddle hook or saddle bag), such variation plays a much smaller role in Sumerian than in Akkadian.

*`italicized text`*## Mount the notebook in Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1 CDLI data acquisition

There are various ways in which one can acquire [CDLI](https://cdli.mpiwg-berlin.mpg.de/) data. The website includes a [Downloads](https://cdli.ucla.edu/?q=downloads) page where one can get access to a daily clone of the catalog and the entire set of transliterations. Alternatively, one can perform a search on the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) search page and request a download of the data (transliteration only or catalog and transliteration data) by pushing a button. This works well for a few or several dozens of texts, but not for very large data sets. The present notebook will download the [CDLI](https://cdli.mpiwg-berlin.mpg.de/) files from the clone on [Github](https://github.com/cdli-gh/data).

### Import Packages
* requests: for communicating with a server over the internet
* tqdm: for creating progress bars
* pandas: data analysis and manipulation; dataframes
* os: for basic Operating System operations (such as creating a directory)

In [ ]:
import requests
from tqdm.auto import tqdm
import pandas as pd
import os

### 1.1 Point to your directory for downloading
In order to download the data to your google drive, you will need to provide the directory path in the code cell below.

In [ ]:
!ls -R "/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI"
#os.makedirs('cdlidata', exist_ok = True)
#data = pd.read_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/transliteration.csv')

'/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI':
 cdli2cuneiform.ipynb		    Headers_megacatalogue_cdli_cat.csv
 cdliatf_unblocked.atf		    megacatalogue_short.csv
 cdli_cat.csv			    megacatalogue_short.gsheet
'cdli_cat_material (1).gsheet'	    processed.csv
 cdli_cat_material.csv		    transliteration.csv
 cdli_cat_material.gsheet	    transliteration.gsheet
 cdli_cat_short.csv		    transliteration.tsv
 cdli_cat_short.gsheet		    Wikidata_cuneiform_signs.csv
'Copy of 2.preprocess_text.ipynb'   Wikidata_cuneiform_signs.gsheet


### 1.2 Download
Because of the size of the files, and depending on the speed of your computer and internet connection the downloading process can take some time.

In [ ]:
CHUNK = 1024
urls = ['https://github.com/cdli-gh/data/raw/master/cdli_cat.csv', 'https://github.com/cdli-gh/data/raw/master/cdliatf_unblocked.atf']
for url in urls:
    target = url.split('/')[-1]
    with requests.get(url, stream=True) as r:
        if r.status_code == 200:
            total_size = int(r.headers.get('content-length', 0))
            tqdm.write(f'Saving {url} as CDLI/{target}')
            t=tqdm(total=total_size, unit='B', unit_scale=True, desc = target)
            with open(f'/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/{target}', 'wb') as f:
                for c in r.iter_content(chunk_size=CHUNK):
                    t.update(len(c))
                    f.write(c)
        else:
            print(f"{url} does not exist.")

Saving https://github.com/cdli-gh/data/raw/master/cdli_cat.csv as CDLI/cdli_cat.csv


cdli_cat.csv:   0%|          | 0.00/155M [00:00<?, ?B/s]

Saving https://github.com/cdli-gh/data/raw/master/cdliatf_unblocked.atf as CDLI/cdliatf_unblocked.atf


cdliatf_unblocked.atf:   0%|          | 0.00/86.9M [00:00<?, ?B/s]

### 1.3 Load in a Pandas data frame

The field `id_text` holds the text ID number as a string, without the preceding "P" and without padding zeroes to the left. The text ID "P001023" is thus represented as 1023. When reading the data into `pandas`, chances are that the data type of `id_text` is interpreted as integer. The function `zfill()` adds the padding zeros to create a six-digit number as a string.

In [ ]:
cat = pd.read_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/cdli_cat.csv', engine='python').fillna('')
cat['id_text'] = ["P" + str(no).zfill(6) for no in cat['id_text']]
cat

,accession_no,accounting_period,acquisition_history,alternative_years,ark_number,atf_source,atf_up,author,author_remarks,cdli_collation,...,seal_information,stratigraphic_level,subgenre,subgenre_remarks,surface_preservation,text_remarks,thickness,translation_source,width,object_remarks
0,,,,,21198/zz001q0dtm,"Englund, Robert K.",,CDLI,"31x61x18; Lú A 14-16.30-32.48-50; M XVIII, auf...",,...,,,Archaic Lu2 A (witness),,,,18,no translation,61,
1,,,,,21198/zz001q0dv4,"Englund, Robert K.",,CDLI,30x48x13; Lú A 13-15.23-25.?; Fundstelle wie W...,,...,,,Archaic Lu2 A (witness),,,,13,no translation,48,
2,,,,,21198/zz001q0dwn,"Englund, Robert K.",,"Englund, Robert K. & Nissen, Hans J.","42x53x19; Vocabulary 9; Qa XVI,2, unter der Ab...",,...,,,Archaic Vocabulary (witness),Text category: 15-09; Foreign ID: LVO 9,,,19,no translation,53,
3,,,,,21198/zz001q0dx5,"Englund, Robert K.",,CDLI,26x23x23; Lú A 9-10.?.?; Fundstelle wie W 9123...,,...,,,Archaic Lu2 A (witness),,,,23,no translation,23,
4,,,,,21198/zz001q0dzp,"Englund, Robert K.",,CDLI,"29x36x20; Lú A Vorläufer; Qa XVI,2, unter der ...",,...,,,Archaic Lu2 A (witness),,,,20,no translation,36,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353278,,,,,,no atf,,"Fahad, Saad Salman & Al-Hussainy, Abbas A.",,,...,,,,,,,,no translation,,
353279,,,,,,no atf,,"Postgate, J. Nicholas",,,...,,,,,,,20,no translation,34,
353280,,,,,,no atf,,"Postgate, J. Nicholas",,,...,,,,,,,20,no translation,34,
353281,,,"purchased from M. Gejou, Paris, in the summer ...",,,no atf,,"Grant, Elihu",,,...,,,,,,,,no translation,,


### 1.4 Use the Catalog to Select Transliterations
In the example code in the following cell the catalog is used to select from the transliteration file all texts from the Early Dynastic IIIa period. The field "period" is used to select those catalog entries that have "ED IIIa" in that field.

In the transliteration file, a new text is introduced by a line that begins with an ampersand (&) followed by a P number, followed by a publication reference (journal or book) using a commonly used set of abbreviations, as in:

> 	&P212416 = AAICAB 1/1, pl. 008, 19282-439

The set of transliterations in the file `cdliatf_unblocked.atf` is read into a list, one line at a time, with the `readlines()` method. The code iterates through that list of lines. The flag `keep` (which initially is set to `False`) is set to `True` if the code encounters a P number that is present in the list `pnos`. As long as `keep = True` subsequent lines are added to the list `ed3a_atf`. When the script encounters a P-number that is not in `pnos`, the flag `keep` is set to `False`.

The result is a of list lines with all the transliteration data of the Early Dynastic IIIa texts in [CDLI](https://cdli.mpiwg-berlin.mpg.de/).

In [ ]:
#atf = pd.read_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/cdliatf_unblocked.atf',encoding="utf8").fillna('')
#atf['id_text'] = ["P" + str(no).zfill(6) for no in atf['id_text']]
#atf

ed3a = cat.loc[cat["period"].str[:7] == "ED IIIa"]
pnos = list(ed3a["id_text"])
#pnos = ["P" + str(no).zfill(6) for no in pnos]
with open("/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/cdliatf_unblocked.atf", encoding="utf8") as c:
    lines = c.readlines()
keep = False
ed3a_atf = []
for line in tqdm(lines):
    if line[0] == "&":
        if line[1:8] in pnos:
            keep = True
        else:
            keep = False
    if keep:
        ed3a_atf.append(line)

  0%|          | 0/3559111 [00:00<?, ?it/s]

### 1.5 Create a Data Frame with the Selected Texts
The following code will transform the list `ed3a_atf` into a format where each text is a row in a `pandas` data frame, with the text ID in column 1, and the transliteration in column 2 (as a single string, without line numbers or line demarcations). This is, of course, just one example of how the data may be selected and formatted - we can use all the power of the `pandas` library to slice and manipulated the data.

```{figure} ../images/P212416.jpg
:scale: 25%
[P212416](https://cdli.mpiwg-berlin.mpg.de//P212416), an ED IIIa administrative document from Kish
```

The lines are read in reverse order, so that when the script encounters an '&P' line (as in '&P212416 = AAICAB 1/1, pl. 008, 19282-439'), this signals that all the lines of a text have been read and that the document can be added to the list `docs`. (When reading the lines in regular order - taking the '&P' line as signaling the end of the previous document - one needs to separately save the last document, because there is no '&P' line anymore to indicate that the text is complete).

In [ ]:
docs = []
document = ''
id_text = ''
ed3a_atf = [line for line in ed3a_atf if line.strip()]  # remove empty lines, which cause trouble
for line in tqdm(reversed(ed3a_atf)):
    if line[0] == "&":  # line beginning with & marks the beginning of a document
        id_text = line[1:8] # retrieve the P number
        docs.append([id_text, document])
        document = ''   # after appending the data to docs, reset the variable `document`.
        continue
    elif line [0] in ["#", "$", "<", ">", "@"]:  # skip all non-transliteration lines
        continue
    else:
        try:
            line = line.split(' ', 1)[1].strip() # split line at first space (after the line number)
            document = f'{line} {document}' # add the new line in front
        except:
            continue   # malformed lines (no proper separation between line number and text) are skipped
ed3a_df = pd.DataFrame(docs)
ed3a_df.columns = ["id_text", "transliteration"]

0it [00:00, ?it/s]

In [ ]:
ed3a_df

,id_text,transliteration
0,P306736,[x e2] sar [n] ku3 ma-na sa10#-bi 1(u@c) ku3 g...
1,P010800,1(asz@c) HUL?-x 1(asz@c)#? ESZ#-ga-x? 1(asz@c)...
2,P010644,LAK358-nu-ru gir2 AB si urin am6-dar sza3 ku3-...
3,P519406,1(u@c) ku3 luh-ha gin2 sa10 e2 2(asz@c) e2-bi ...
4,P519401,[...] x [...] [...] x [...] x-gal# [...] tu7(|...
...,...,...
1009,P010012,1(esze3@c) GAN2 ur-{d}nin-PA-ke4 lugal-USZ-MUS...
1010,P010011,6(asz@c) uruda ma-na sa10 GAN2 4(iku@c) GAN2-b...
1011,P010009,[6(asz@c)] uruda ma-na sa10 GAN2 4(iku@c) GAN2...
1012,P010008,4(iku@c) GAN2 DUR2-HAR sar 1(u@c) uruda <a>-ru...


###1.6 Complete for all texts in the dataset: cdli_atf
* save as 'cdli_atf.csv'

| pd_index | id_text | transliteration |
| -------- | ------- | --------------- |
| 0 | P306736 | [x e2] sar [n] ku3 ma-na sa10#-bi 1(u@c) ku3 g... |

In [ ]:
# save as cdli_atf
ed3a_df.to_csv('cdli_atf.csv', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from fastprogress.fastprogress import progress_bar

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##2 Sign List Dictionary
The nexts steps include the creation of a dictionary which we can use for matching sign readings to unicode cuneiform characters. We can also update with any unmatched signs to ensure we have a complete dictionary not only for the CDLI, but any other reading of a cuneiform sign.

###2.1 Nuolenna Sign List
1. We can use Timo's version of the Nuolenna sign list:

https://github.com/situx/Nuolenna/blob/master/sign_list.json

We may want to make some changes and additions as we go.

* To open the sign_list.json

* You need to modify the URL to obtain the raw CSV file content. Here's how you can do it:

* Open the URL in your web browser: https://github.com/situx/Nuolenna/blob/master/sign_list.json
* Click on the "Raw" button, which should display the raw content of the JSON file.
* Copy the URL from the address bar of your web browser, which should look something like this: https://raw.githubusercontent.com/situx/Nuolenna/master/sign_list.json
* By using the raw URL, you will be directly accessing the CSV file's content, allowing pandas to parse it correctly.

  Code:

  `import pandas as pd`

  `sign_list = pd.read_csv('https://raw.githubusercontent.com/situx/Nuolenna/master/sign_list.json')`

2. Make a DataFrame for the Nuolenna sign list:

| pd_index | Sign | Unicode |
| -------- | ---- | ---- |
| 0 | / | 𒑰 |
| 1 | /ad/ | 𒈕 |


In [ ]:
# Reading in the signlist from json format
sign_list = pd.read_json('https://raw.githubusercontent.com/situx/Nuolenna/master/sign_list.json', orient='index')

In [ ]:
# name the columns, cleaning, rearranging
sign_list.columns = ['unicode']
form = sign_list.index.tolist()
sign_list['sign'] = form
desired_order = ['sign', 'unicode']
sign_list = sign_list[desired_order]
sign_list = sign_list.reset_index(drop=True)

In [ ]:
# now have columns named "index", "Sign", "Unicode"
sign_list

,sign,unicode
0,(4×za)×kur,𒍟
1,(an.naga)@(an.naga),𒀰
2,(bu&bu)×na₂,𒒈
3,(kaskal.lagab×u)&(kaskal.lagab×u),𒆞
4,(muš&muš)×ga,𒔤
...,...,...
12612,ṭu₃,𒁺
12613,ṭu₄,𒌈
12614,ṭu₅,𒃮
12615,ṭu₆,𒅲


### 2.2 Akkademia Sign List
Check against this sign list: https://github.com/gaigutherz/Akkademia/blob/master/cuneiform_to_unicode_fixed.csv

(The format shoudl be the same...)

* Evaluate & Check with Akkademia Sign List:
* compare Nuolenna sign_list.json against the Akkademia sign list
* create a new dataframe with all the signs and values in CSV format



In [ ]:
akkademia = pd.read_csv('https://raw.githubusercontent.com/gaigutherz/Akkademia/master/cuneiform_to_unicode_fixed.csv')
akkademia

,sign,unicode
0,ʾu₄,𒀀
1,'u4,𒀀
2,a,𒀀
3,aia₂,𒀀
4,aia2,𒀀
...,...,...
14236,/,𒑰
14237,:,𒑱
14238,":""",𒑲
14239,:.,𒑳


In [ ]:
concatenated = pd.merge(sign_list, akkademia, on=['sign', 'unicode'], how='outer', indicator=True)
concatenated['_merge'] = concatenated['_merge'].replace({'left_only': 'sign_list', 'right_only': 'akkademia'})
non_overlapping_rows = concatenated[concatenated['_merge'] != 'both']


In [ ]:
concatenated

,sign,unicode,_merge
0,(4×za)×kur,𒍟,sign_list
1,(an.naga)@(an.naga),𒀰,sign_list
2,(bu&bu)×na₂,𒒈,sign_list
3,(kaskal.lagab×u)&(kaskal.lagab×u),𒆞,sign_list
4,(muš&muš)×ga,𒔤,sign_list
...,...,...,...
18192,6(bur3),𒐑,akkademia
18193,7(bur3),𒐒,akkademia
18194,8(bur3),𒐓,akkademia
18195,9(bur3),𒐔,akkademia


In [ ]:
non_overlapping_rows

,sign,unicode,_merge
0,(4×za)×kur,𒍟,sign_list
1,(an.naga)@(an.naga),𒀰,sign_list
2,(bu&bu)×na₂,𒒈,sign_list
3,(kaskal.lagab×u)&(kaskal.lagab×u),𒆞,sign_list
4,(muš&muš)×ga,𒔤,sign_list
...,...,...,...
18192,6(bur3),𒐑,akkademia
18193,7(bur3),𒐒,akkademia
18194,8(bur3),𒐓,akkademia
18195,9(bur3),𒐔,akkademia


###2.2.1 Adding problem signs to the list:

We will want to add the following signs to the list:

* kux(DU) = 𒁺
* ARAD = 𒀴
* munx(GAKKUL) = 𒌋𒁵
* isx(US2) = 𒍑
* ah:du = 𒄴𒁺
* kak(NI) = 𒉌
* GAN2 = 𒃷
* HI = 𒄭
* KU = 𒆪
* AZ = 𒊍
* gag(NI) = 𒉌
* AB = 𒀊
* NE = 𒉈
* NA = 𒈾
* AN = 𒀭
* SI A = 𒋛𒀀
* EN = 𒂗
* PA = 𒉺
* DUN = 𒂄
* NIM = 𒉏
* ASZ = 𒀸
* TUL2 = 𒇥
*

Other conversion issues that returned N/A:
* n+3(disz) --> X 𒐈 (rule here is n+ --> X)
*

###2.3. ORACC GLobal Sign List (OGSL)

Even though the dataset has the CDLI formatting, we include the most recent updates from the ORACC Global Sign List:

https://raw.githubusercontent.com/oracc/ogsl/master/00lib/ogsl.asl

https://github.com/oracc/ogsl/commits/master

__Sections__:
0. OGSL import and formatting
  * [Example: A](https://docs.google.com/spreadsheets/d/1-GomqApr-pproRxE4f7PEkXVdbc0sL7aovL-F7RM2xg/edit?usp=sharing)
  * We can use this to build a comparable signlist, which we can check against Nuolenna and Akkademia sign lists.
1. How to address outdated mappings in the list, e.g., meš as 𒈨𒌍 rather than 𒎌 (the latter was added in Unicode 7, in 2014)?

  * Check differences between OGSL and the existing dictionary
  * When there unicode differs beween the dictionary and OGSL, flag the sign.
  * A specialist will check, e.g. 'meš' replace (old) 𒈨𒌍 with (new) 𒎌
2. Adding time periods to each token
  * We can use FactGrid IDs and Wikidata IDs for [LOD Periods](https://docs.google.com/spreadsheets/d/1dXVNI1dEhrLo5Vy_CoFo6WxgV65TZ_Vl7aBpWsY520I/edit?usp=sharing)

|id_text|id_word|period|FG_Q|WD-Q|
|-------|-------|------|----|----|
|P123456|P123456.1|Ur III (ca. 2100-2000 BC)|Q512148|Q109384761|



## 3 Punctuation

Punctuation marks will stand in the way of a conversion to unicode cuneiform. To address this issue, there is a series of steps which must follow a sequential order, so as not to remove meaningful elements from the sign readings.

* See ORACC help guides for summary of compound grapheme operators in ATF:
http://oracc.museum.upenn.edu/doc/help/editinginatf/primer/inlinetutorial/index.html

* Several types of cuneiform punctuation are supported in ATF and all of them must be preceded and followed by a space (in the case of * and / the punctuation may be immediately followed by a sign name in parentheses and then the following space). The recognized punctuation codes are:

1. * = Bullet: The "1" used at the start of each line in lexical texts, omen compendia, etc..
2. *(GRAPHEME): Generic punctuation; most often used where scribes use signs other than a "1" at the start of the line in lexical texts, but may be used to transliterate arbitrary or unusual kinds of punctuation that are not otherwise covered below.
3. : = cuneiform vertical colon. The vertical "colon" sign often found in commentaries. N.B.: If the single colon occurs within a word it must be transliterated with the grapheme name form P₂
4. :' (colon+right-quote) = Borger MZL 592 variant b; a variant on the vertical two-wedge colon
5. :" (colon+double-quote) = cuneiform diagonal colon. The diagonal "colon" sign often found in commentaries. Note that the three different double-wedge colon signs are mnemonically two-dots, two-dots-prime and two-dots-double-prime
6. :. = cuneiform triple wedge colon.  The triple-wedge "colon" sign sometimes found in commentaries.
7. :: = ?? (A colon convention defined in the SAA style manual, form unspecified.)
8. / = Word divider; if unqualified, this is the single vertical wedge word-divider as used, e.g., in Old Assyrian texts. May be qualified as, e.g., /(P2).



###3.1 Punctuation removal (in sequence)

1. Remove curley brackets '{ }' (one space added)
2. Remove hyphen '-' (one space added)
3. Remove: _, [, ], *, !, ?, /, \, :, ;, ^, `,  (no spaces added)
4. Retain: ., |, (, ), #, ~, @, %, $, '...', =, <, >, x,
* Adam will review the reulting list of signs that do not get matched to a cuneiform sign. For this we want a CSV of unique tokens (with punctuation).


###3.2 Allographs and Compund Grapheme Modifiers

Allographs use the tilde `~` which we need to remove for proper matching. However the characters/numbers following the tilde are important to remove as well. In doing so, it is important to note the case change e.g.`GEŠTU~axŠE~a@t` = GEŠTU ŠE
  * Example: ~a, ~a2, ~b, ~c, ~d, ~d1, ~e?, ~h, ~w
  * you can see that there are sometimes numbers following the character, directly after the tilde: ~a**2**  
  * see [ATF Inline Tutorial](http://oracc.museum.upenn.edu/doc/help/editinginatf/primer/inlinetutorial/index.html)

---------------------------------------------------

####Summary of Compound Grapheme Operators in ATF/GDL:

* beside	. =	`|DU.DU|`
* joining	+	= `|LAGAB+LAGAB|`
* containing	×	= `|GA₂×AN|`	(GA TWO TIMES AN)
* containing/group	× =	`|GA₂×(ME.EN)|`	(GA TWO TIMES ME PLUS EN)
* above	& =	`|DU&DU|` (DU OVER DU)
* crossing	%	= `|GI%GI|`	(GI CROSSING GI)
* opposing	@	= `|LU₂@LU₂|`	(LU TWO OPPOSING LU TWO)
* repeated	3×	= `|3×AN|`	(THREE TIMES AN)
* repeated	4×	= `|4xLU2|`	(FOUR TIMES LU TWO)


In [ ]:
cdli_unicode = pd.read_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/transliteration.csv')
cdli_unicode

,id_text,transliteration
0,P496727,2(disz) me _udu hi-a_ sza su-ga-gu-ut {disz}me...
1,P353473,1(disz) me 1(u) _udu hi-a_ sza su-ga-gu-ut a-a...
2,P496726,i-na 1(disz) ma-na ku3-babbar_ sza su-ga-gu#-u...
3,P496725,i-na 1(disz) me _udu hi-a_ sza su-ga-gu-ut ha-...
4,P496724,i-na _3(disz) 1/3(disz) ma-na ku3-babbar_ 2(di...
...,...,...
132946,P000005,"1(N01) , ESZDA# 1(N01) , KAB 1(N01) , DI@t? 1(..."
132947,P000004,"1(N01) , |SZE~a.NAM2|# 1(N01) , PA~a#? [...] 1..."
132948,P000003,"1(N01) , NI~b# 1(N01) , |NI~bxX| 1(N01) , IR~a..."
132949,P000002,"[1(N01)] , [...] [1(N01)] , GAL~a# SZITA~a1 [1..."


In [ ]:
#this format is for unicode
to_unicode = cdli_unicode.copy()

In [ ]:
#1 check sign

to_unicode['transliteration original'] = to_unicode['transliteration']

to_unicode['transliteration'] = to_unicode['transliteration'].astype(str).fillna('')


# Sign to check for existence
sign_to_check = '……'

# Method 1: Using the "any" method with a lambda function
exists = to_unicode.iloc[:, 1].apply(lambda x: sign_to_check in x).any()

# Method 2: Using the "str.contains" method
exists = to_unicode.iloc[:, 1].str.contains(sign_to_check).any()

# Check if the sign exists in the second column
if exists:
    print(f"The sign '{sign_to_check}' exists in the second column.")
else:
    print(f"The sign '{sign_to_check}' does not exist in the second column.")


The sign '……' does not exist in the second column.


In [ ]:
#2 Replacing spaces with any desired character (e.g., underscore '_')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(' ', '……')

In [ ]:
# Old method

#3 Remove underscore
# to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('_', '')

In [ ]:
#get a copy for transliteration format
to_transliteration = to_unicode.copy()

In [ ]:
# Old method

# #4 replace hyphen with space
# to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('-', ' ')

In [ ]:
# Old method

# #5 remove the curly-brackets { } and add a white space before and after
# to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('{', ' ')
# to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('}', ' ')

In [ ]:
# Old method

#6 remove the other punctuation marks, e.g. #, ?, !, others?
# finding special characters
# import re

# special_characters = r'[!\"#$%&\'*+,./:;<=>?@[\\\]^`|~…]'
# special_chars_list  = to_unicode['transliteration'].apply(lambda x: re.findall(special_characters, x))
# unique_special_chars = set(item for sublist in special_chars_list for item in sublist)
# unique_special_chars

In [ ]:
#1. Remove curley brackets '{ }' (one space added)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('{', ' ')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('}', ' ')

<ipython-input-52-4c2a1e88adc4>:2: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('{', ' ')
<ipython-input-52-4c2a1e88adc4>:3: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('}', ' ')


In [ ]:
#2. Add special character for word dividers: 、
#replace a space with separation sign
to_transliteration['transliteration'] = to_transliteration['transliteration'].str.replace(' ', ' 、 ')

In [ ]:
#3 replace hyphen with space
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('-', ' ')

In [ ]:
#4. Remove: _, [, ], *, !, ?, /, , :, ;, ^, `, (no spaces added)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('_', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('[', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(']', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('*', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('!', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('?', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('/', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(',', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(':', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(';', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('^', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('`', '')

# _, [, ], *, !, ?, /, , :, ;, ^, `, (no spaces added)

<ipython-input-55-f039f9464bb5>:3: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('[', '')
<ipython-input-55-f039f9464bb5>:4: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(']', '')
<ipython-input-55-f039f9464bb5>:5: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('*', '')
<ipython-input-55-f039f9464bb5>:7:

In [ ]:
#remove special chars with space(other than '(',')')
# unique_special_chars.remove('(')
# unique_special_chars.remove(')')
# for char in special_characters:
#     cdli_unicode['transliteration'] = cdli_unicode['transliteration'].str.replace(char, '')


In [ ]:
#7 a little unsure yet

In [ ]:
#8 lower case
to_unicode['transliteration'] = to_unicode['transliteration'].str.lower()

In [ ]:
# additional rules:
# 1 Remove / delete the following: erased, ~a, ~b, ~c, ~t, (anything immediately following the tilde ~)
# Function to remove "~" and anything following it until "……" appears
def remove_tilde_and_following(df_column):
    return df_column.str.replace(r'~.*?……', '……')

# Apply the function to the desired column
to_unicode['transliteration'] = remove_tilde_and_following(to_unicode['transliteration'])


# if we were to remove "X"
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('X', '')


<ipython-input-59-c182dcdd1126>:5: FutureWarning: The default value of regex will change from True to False in a future version.
  return df_column.str.replace(r'~.*?……', '……')


In [ ]:
#9 change …… sign back to space
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('……', ' ')

In [ ]:
def extract_and_remove_from_original(df_column):
    extracted_words = df_column.str.extractall(r'\|([^|]*)\|')
    extracted_words.reset_index(inplace=True, drop=True)

    # Remove the extracted words from the original DataFrame
    df_column = df_column.str.replace(r'\|[^|]*\|', '', regex=True)

    return extracted_words

# Create a new DataFrame to store the extracted words
pipe_dataframe = extract_and_remove_from_original(to_unicode['transliteration'])

# Convert any NaN values to empty strings
pipe_dataframe = pipe_dataframe.fillna('')

pipe_dataframe['transliteration'] = pipe_dataframe.apply(lambda row: '|'.join(row), axis=1)

# Concatenate the extracted words into a single column
pipe_dataframe['extracted_words'] = pipe_dataframe.apply(lambda row: [word for word in row if word], axis=1)

# Drop duplicate rows
pipe_dataframe = pipe_dataframe.drop_duplicates(subset='extracted_words', keep='first')

# Drop duplicated word
pipe_dataframe['extracted_words'] = pipe_dataframe['extracted_words'].apply(lambda row: [row[0]])

# Drop the extracted words columns
pipe_dataframe = pipe_dataframe.drop(columns=[0, "transliteration"])

pipe_dataframe

,extracted_words
0,[kib.nun]
3,[kusz.lu.ub2.ki.gar]
5,[igi.ir]
6,[ud.ka.bar]
10,[ninda2x(sze.2(asz))]
...,...
25027,[sze 1(n01) pap 1(n01) ]
25028,[gurusz 1(n01) szita 1(n01) kisal nun 1(n01)...
25029,[ 1(n01) dara3 1(n01) din 1(n01) guruszda# ...
25030,[ni 1(n01) ir 1(n01) bir 1(n01) bir 1(n01) ...


##4 Match sign readings to unicode cuneiform

Using the dictionary from dataframe A, map the sign readings to unicode equivalents.

Make a dataframe with the following fields:

| id_text | transliteration | unicode |
| ------- | --------------- | ------- |
| P496727 | 2(disz) me udu hi-a sza su-ga-gu-ut ... | (unicode signs) |


In [ ]:
#10 convert to sign
# Create a dictionary from dataframe A mapping signs to unicode
unicode_dict = dict(zip(concatenated['sign'], concatenated['unicode']))

def replace_with_unicode(text):
    unicode_values = []
    for sign in text.split():
        unicode = unicode_dict.get(sign)
        if unicode is not None:
            unicode_values.append(unicode)
        else:
            unicode_values.append(sign)  # Keep the sign if no corresponding unicode value found
    return ' '.join(unicode_values)

to_unicode['unicode'] = to_unicode['transliteration'].apply(replace_with_unicode)

# Second time filtering speical characters
# def remove_special_characters(text):
#     cleaned_text = re.sub(r'[!?#.\[\]]', '', text)
#     return cleaned_text

# Apply the function to the 'sentences' column
# to_unicode['unicode'] = to_unicode['unicode'].apply(remove_special_characters)

# to_unicode['unicode'] = to_unicode['unicode'].apply(replace_with_unicode)
to_unicode

# if we were to remove "X"
# to_unicode['unicode'] = to_unicode['unicode'].str.replace('X', '')
to_unicode

,id_text,transliteration,transliteration original,unicode
0,P496727,2(disz) me udu hi a sza su ga gu ut disz me e...,2(disz) me _udu hi-a_ sza su-ga-gu-ut {disz}me...,𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒁹 𒈨 𒀉 𒈪 𒅎 𒇽 𒅀 𒌝 𒄩 𒈪 𒄿 𒊭 𒀀 ...
1,P353473,1(disz) me 1(u) udu hi a sza su ga gu ut a an ...,1(disz) me 1(u) _udu hi-a_ sza su-ga-gu-ut a-a...,𒁹 𒈨 𒌋 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒀀 𒀭 𒇷 𒅎 𒁹 𒈨 𒇻 𒀴 𒊭 𒋢 𒂵 𒄖 ...
2,P496726,i na 1(disz) ma na ku3 babbar sza su ga gu# ut...,i-na 1(disz) ma-na ku3-babbar_ sza su-ga-gu#-u...,𒄿 𒈾 𒁹 𒈠 𒈾 𒆬 𒌓 𒊭 𒋢 𒂵 gu# ut# 𒄩 𒉌 𒀭 𒅎 𒇽 sza# 𒄠 𒁕...
3,P496725,i na 1(disz) me udu hi a sza su ga gu ut ha ad...,i-na 1(disz) me _udu hi-a_ sza su-ga-gu-ut ha-...,𒄿 𒈾 𒁹 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒄩 𒀜 𒉌 𒀭 𒅎 𒇽 𒊭 𒄠 𒁺 𒁲 𒄿 ...
4,P496724,i na 3(disz) 13(disz) ma na ku3 babbar 2(disz)...,i-na _3(disz) 1/3(disz) ma-na ku3-babbar_ 2(di...,𒄿 𒈾 𒐈 13(disz) 𒈠 𒈾 𒆬 𒌓 𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒄩 𒀜...
...,...,...,...,...
132946,P000005,1(n01) eszda# 1(n01) kab 1(n01) di@t 1(n01)...,"1(N01) , ESZDA# 1(N01) , KAB 1(N01) , DI@t? 1(...",1(n01) eszda# 1(n01) 𒆏 1(n01) di@t 1(n01) 𒂗 1(...
132947,P000004,1(n01) |sze 1(n01) pa ... 1(n01) ... 1(n01)...,"1(N01) , |SZE~a.NAM2|# 1(N01) , PA~a#? [...] 1...",1(n01) |sze 1(n01) 𒉺 ... 1(n01) ... 1(n01) ...
132948,P000003,1(n01) ni 1(n01) |ni 1(n01) ir 1(n01) bir ...,"1(N01) , NI~b# 1(N01) , |NI~bxX| 1(N01) , IR~a...",1(n01) 𒉌 1(n01) |ni 1(n01) 𒅕 1(n01) 𒄵 1(n01) 𒄵...
132949,P000002,1(n01) ... 1(n01) gal szita 1(n01) abgal# 1...,"[1(N01)] , [...] [1(N01)] , GAL~a# SZITA~a1 [1...",1(n01) ... 1(n01) 𒃲 𒋖 1(n01) abgal# 1(n01) ......


In [ ]:
# to_transliteration.head(1)
# to_transliteration['unicode'] = to_transliteration['unicode'].str.replace(' ', '')
# #9 change ， sign back to space
# to_transliteration['unicode'] = to_transliteration['unicode'].str.replace('、', ' ')

In [ ]:
# this is the code for sorting out the code that did not get converted to unicode signs
import re

alphabetical = r'\b[A-Za-z]+\b'
alphabetical_list = to_unicode['unicode'].apply(lambda x: re.findall(alphabetical, x))
alphabetical_list

series_data_reset = alphabetical_list.reset_index()

series_data_reset.columns = ['Row Number', 'words']

temp = series_data_reset.melt(id_vars='Row Number', value_vars='words')

filtered_df = temp[['value']]

failed_converted_words = filtered_df.rename(columns={'value': 'words'})

failed_converted_words = filtered_df[filtered_df['value'].apply(lambda x: len(x) > 0)]

failed_converted_words = pd.merge(failed_converted_words, to_unicode, left_index=True, right_index=True).drop(["transliteration","unicode","id_text"],  axis=1)

In [ ]:
failed_converted_words

,value,transliteration original,roman,count
0,"[zi, im, ma]",2(disz) me _udu hi-a_ sza su-ga-gu-ut {disz}me...,True,4
1,"[ii, ia, u, udu, szu, ud, du, li, im, ma]",1(disz) me 1(u) _udu hi-a_ sza su-ga-gu-ut a-a...,True,9
2,"[gu, ut, sza, im, disz, tum, lugal, si, disz, ...",i-na 1(disz) ma-na ku3-babbar_ sza su-ga-gu#-u...,True,23
4,"[disz, u, udu, hi, a, hi, ir, disz, ma]",i-na _3(disz) 1/3(disz) ma-na ku3-babbar_ 2(di...,True,10
5,"[X, blank, space, X, X, X, a, na, sza, nu, X, ...",[...]-tim [...]-x{ki} [($ blank space $) x] x ...,True,77
...,...,...,...,...
132946,"[eszda, di, t, X, X]","1(N01) , ESZDA# 1(N01) , KAB 1(N01) , DI@t? 1(...",True,14
132947,[sze],"1(N01) , |SZE~a.NAM2|# 1(N01) , PA~a#? [...] 1...",True,5
132948,"[ni, X, nu, g, X, lagab, lagab, X, X, X, X]","1(N01) , NI~b# 1(N01) , |NI~bxX| 1(N01) , IR~a...",True,41
132949,"[abgal, X]","[1(N01)] , [...] [1(N01)] , GAL~a# SZITA~a1 [1...",True,9


##5 Evaluation for Resulting Unmatched Sign Readings

Next we make a list of the unmatched signs. In order to evaluate why they did not find a match, we will want to include the original format for each sign in the extracted words (above), along with the cleaned text as it appears in the above dataframe 'value'.Each item should also include the 'id_text' they belong to. This CSV can be used to supervise the unmatched signs and will aid in the development of the signs which are not yet included in unicode.

###Example of format for export to CSV:

|id_text|alphabetical|cleaned|original|
|--|--|--|--|
|P000004| nam2 | sze nam2 | \|SZE~a.NAM2\|# |

### 5.1 Export Unmatched to CSV

In [ ]:

#failed_converted_words.to_csv('/content/drive/MyDrive/Harrison/alphabetical.csv')
failed_converted_words.to_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Harrison/unmatched.csv')

###5.2 Assessment of unicode conversion

1. How many rows (unique id_text) still have roman alphabet characters? [a-z]
2. Can we count how many roman alphabet characters there are for this subset?

|id_text|transliteration|unicode|roman|count|
|--|--|-------|-----|-----|
|P496727|2(disz) me udu hi a sza su ga gu ut disz me e...	|𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒁹 𒈨 𒀉 𒈪 𒅎 𒇽 𒅀 𒌝 𒄩 𒈪 𒄿 𒊭 𒀀 ...|False|0|
|P000001|1(n01) , [...] 1(n01) , tim abgal# 1(n01) , ki...	|1(n01) , 𒈀𒇲 1(n01) , 𒁴 𒉣𒈨 1(n01) , 𒃲𒌺 1(n01) ,...|True|4|

In [ ]:
# Function to check if a word has Roman alphabet
def has_roman_alphabet(word):
    return bool(re.search(r'[a-zA-Z]', word))

# Function to count the number of words with Roman alphabet in a sentence
def count_roman_words(sentence):
    words = sentence.split()
    return sum(has_roman_alphabet(word) for word in words)

to_unicode['roman'] = to_unicode['unicode'].apply(lambda x: any(has_roman_alphabet(word) for word in x.split()))
to_unicode['count'] = to_unicode['unicode'].apply(count_roman_words)

to_unicode

,id_text,transliteration,transliteration original,unicode,roman,count
0,P496727,2(disz) me udu hi a sza su ga gu ut disz me e...,2(disz) me _udu hi-a_ sza su-ga-gu-ut {disz}me...,𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒁹 𒈨 𒀉 𒈪 𒅎 𒇽 𒅀 𒌝 𒄩 𒈪 𒄿 𒊭 𒀀 ...,True,4
1,P353473,1(disz) me 1(u) udu hi a sza su ga gu ut a an ...,1(disz) me 1(u) _udu hi-a_ sza su-ga-gu-ut a-a...,𒁹 𒈨 𒌋 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒀀 𒀭 𒇷 𒅎 𒁹 𒈨 𒇻 𒀴 𒊭 𒋢 𒂵 𒄖 ...,True,9
2,P496726,i na 1(disz) ma na ku3 babbar sza su ga gu# ut...,i-na 1(disz) ma-na ku3-babbar_ sza su-ga-gu#-u...,𒄿 𒈾 𒁹 𒈠 𒈾 𒆬 𒌓 𒊭 𒋢 𒂵 gu# ut# 𒄩 𒉌 𒀭 𒅎 𒇽 sza# 𒄠 𒁕...,True,23
3,P496725,i na 1(disz) me udu hi a sza su ga gu ut ha ad...,i-na 1(disz) me _udu hi-a_ sza su-ga-gu-ut ha-...,𒄿 𒈾 𒁹 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒄩 𒀜 𒉌 𒀭 𒅎 𒇽 𒊭 𒄠 𒁺 𒁲 𒄿 ...,False,0
4,P496724,i na 3(disz) 13(disz) ma na ku3 babbar 2(disz)...,i-na _3(disz) 1/3(disz) ma-na ku3-babbar_ 2(di...,𒄿 𒈾 𒐈 13(disz) 𒈠 𒈾 𒆬 𒌓 𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒄩 𒀜...,True,10
...,...,...,...,...,...,...
132946,P000005,1(n01) eszda# 1(n01) kab 1(n01) di@t 1(n01)...,"1(N01) , ESZDA# 1(N01) , KAB 1(N01) , DI@t? 1(...",1(n01) eszda# 1(n01) 𒆏 1(n01) di@t 1(n01) 𒂗 1(...,True,14
132947,P000004,1(n01) |sze 1(n01) pa ... 1(n01) ... 1(n01)...,"1(N01) , |SZE~a.NAM2|# 1(N01) , PA~a#? [...] 1...",1(n01) |sze 1(n01) 𒉺 ... 1(n01) ... 1(n01) ...,True,5
132948,P000003,1(n01) ni 1(n01) |ni 1(n01) ir 1(n01) bir ...,"1(N01) , NI~b# 1(N01) , |NI~bxX| 1(N01) , IR~a...",1(n01) 𒉌 1(n01) |ni 1(n01) 𒅕 1(n01) 𒄵 1(n01) 𒄵...,True,41
132949,P000002,1(n01) ... 1(n01) gal szita 1(n01) abgal# 1...,"[1(N01)] , [...] [1(N01)] , GAL~a# SZITA~a1 [1...",1(n01) ... 1(n01) 𒃲 𒋖 1(n01) abgal# 1(n01) ......,True,9


###5.3 Export Full List of Strings to CSV

* CDLI_strings.csv

In [ ]:
#to_unicode.to_csv('/content/drive/MyDrive/Harrison/CDLI_strings.csv')
to_unicode.to_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Harrison/CDLI_strings.csv')

In [ ]:
#example show
to_unicode.head(5)

,id_text,transliteration,transliteration original,unicode,roman,count
0,P496727,2(disz) me udu hi a sza su ga gu ut disz me e...,2(disz) me _udu hi-a_ sza su-ga-gu-ut {disz}me...,𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒁹 𒈨 𒀉 𒈪 𒅎 𒇽 𒅀 𒌝 𒄩 𒈪 𒄿 𒊭 𒀀 ...,True,4
1,P353473,1(disz) me 1(u) udu hi a sza su ga gu ut a an ...,1(disz) me 1(u) _udu hi-a_ sza su-ga-gu-ut a-a...,𒁹 𒈨 𒌋 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒀀 𒀭 𒇷 𒅎 𒁹 𒈨 𒇻 𒀴 𒊭 𒋢 𒂵 𒄖 ...,True,9
2,P496726,i na 1(disz) ma na ku3 babbar sza su ga gu# ut...,i-na 1(disz) ma-na ku3-babbar_ sza su-ga-gu#-u...,𒄿 𒈾 𒁹 𒈠 𒈾 𒆬 𒌓 𒊭 𒋢 𒂵 gu# ut# 𒄩 𒉌 𒀭 𒅎 𒇽 sza# 𒄠 𒁕...,True,23
3,P496725,i na 1(disz) me udu hi a sza su ga gu ut ha ad...,i-na 1(disz) me _udu hi-a_ sza su-ga-gu-ut ha-...,𒄿 𒈾 𒁹 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒄩 𒀜 𒉌 𒀭 𒅎 𒇽 𒊭 𒄠 𒁺 𒁲 𒄿 ...,False,0
4,P496724,i na 3(disz) 13(disz) ma na ku3 babbar 2(disz)...,i-na _3(disz) 1/3(disz) ma-na ku3-babbar_ 2(di...,𒄿 𒈾 𒐈 13(disz) 𒈠 𒈾 𒆬 𒌓 𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒄩 𒀜...,True,10


In [ ]:
#Count how many rows are useful
count_zero_rows = (to_unicode['count'] == 0).sum()
count_less_than_five = (to_unicode['count'] <= 5).sum()
print("Number of rows with count less than 5:", count_less_than_five)
print("Number of rows with count 0:", count_zero_rows)

Number of rows with count less than 5: 95945
Number of rows with count 0: 44452


In [ ]:
#Filter to only count <= 5 rows
less_than_or_equal_to_5_roman = to_unicode[to_unicode['count'] <= 5]

less_than_or_equal_to_5_roman

,id_text,transliteration,transliteration original,unicode,roman,count
0,P496727,2(disz) me udu hi a sza su ga gu ut disz me e...,2(disz) me _udu hi-a_ sza su-ga-gu-ut {disz}me...,𒈫 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒁹 𒈨 𒀉 𒈪 𒅎 𒇽 𒅀 𒌝 𒄩 𒈪 𒄿 𒊭 𒀀 ...,True,4
3,P496725,i na 1(disz) me udu hi a sza su ga gu ut ha ad...,i-na 1(disz) me _udu hi-a_ sza su-ga-gu-ut ha-...,𒄿 𒈾 𒁹 𒈨 𒇻 𒄭 𒀀 𒊭 𒋢 𒂵 𒄖 𒌓 𒄩 𒀜 𒉌 𒀭 𒅎 𒇽 𒊭 𒄠 𒁺 𒁲 𒄿 ...,False,0
6,P496970,1(disz) gu4 i na an nu ni tim sza li bi a lim ...,1(disz) _gu4_ i-na an-nu-ni-tim sza li-bi a-li...,𒁹 𒄞 𒄿 𒈾 𒀭 𒉡 𒉌 𒁴 𒊭 𒇷 𒁉 𒀀 𒅆 𒁹 𒄿 𒈾 𒀭 𒉡 𒉌 𒁴 𒊭 𒅗 𒉿 ...,False,0
7,P496969,4(disz) udu e2 d nin hur sag ga2 4(disz) udu ...,_4(disz) udu e2_ {d}nin-hur-sag-ga2 _4(disz) u...,𒐉 𒇻 𒂍 𒀭 𒊩𒌆 𒄯 𒊕 𒂷 𒐉 𒇻 𒂍 𒀭 𒁕 𒃶 𒐉 𒇻 𒂍 𒀭 𒄩 na# 𒀜 𒐉...,True,1
13,P496964,2(disz) uzu ma la ku sza gu4 a na ku um zi il...,2(disz) {uzu}ma-la-ku sza _gu4_ a-na ku-um-zi-...,𒈫 𒍜 𒈠 𒆷 𒆪 𒊭 𒄞 𒀀 𒈾 𒆪 𒌝 𒍣 𒅋 𒇷 𒅇 𒊩 𒍣 𒅅 𒊑 𒅎 𒋗 𒋾 𒀀 ...,True,1
...,...,...,...,...,...,...
132925,P000033,1(n01) ku x du6 1(n01) tar du6 1(n01) gal u...,"1(N01) , KU~b2 X DU6~b 1(N01) , TAR~a DU6~b 1(...",1(n01) 𒆪 X 𒇯 1(n01) 𒋻 𒇯 1(n01) 𒃲 𒀖𒆪 X ...,True,5
132929,P000024,1(n01) ... 1(n01) sanga |(bu 1(n01) bur2 .....,"[1(N01)] , [...] 1(N01) , SANGA~a |(BU~a&BU~a)...",1(n01) ... 1(n01) 𒋃 |(bu 1(n01) 𒁔 ... 1(n01) ...,True,5
132942,P000008,1(n01) sila3 isz ... 1(n01) ni tur 1(n01) s...,"[1(N01)] , SILA3~a#? ISZ~b# [...] [1(N01)] , N...",1(n01) 𒋡 𒅖 ... 1(n01) 𒉌 𒌉 1(n01) 𒊩 zatu750,True,4
132945,P292940,1(u) 2(asz) gur sze hu bu ta tum u2 sze ti iq ...,_1(u) 2(asz) gur sze_ hu-bu-ta-tum u2-sze-ti-i...,𒌋 𒐀 𒄥 𒊺 𒄷 𒁍 𒋫 𒌈 𒌑 𒊺 𒋾 𒅅 𒈠 𒀸 𒄥 𒁹 𒑒 𒈧 𒌑 𒊓 𒀊 𒆠 𒀭 ...,True,4


##6 ORACC Format Conversion: `cdli_forms`

In this step we will convert the CDLI strings of sign readings to a data frame where each word token is in a separate row. This will allow for a conversion to unicode that is consistent with the workflow for the ORACC texts as well.

__Steps include:__
0. Make a pd `cdli_forms`
1. Replace spaces with separation sign: `、`
2. Use the separation sign `、` to split each word into a new row in the `cdli_forms` table
3. Add a 'clean' column to the data frame for the pre-processing step before converting to cuneiform
4. Retain the proper `id_text` for each word, along with a count for each word:

|id_text|id_word|form|clean|a-Z|
|--|--|--|--|--|
|P496727|1|2(disz)|||
|P496727|2|me|||
|P496727|3|udu|||
|P496727|4|hi-a|||


In [ ]:
import pandas as pd
import re


# Read the CSV file
cdli_unicode = pd.read_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/transliteration.csv')

# Process data in segments of 1000 rows
chunk_size = 1000
total_rows = len(cdli_unicode)

# number of rows can change for shorter runs, e.g. (0, total_rows, chunk_size) > (0, 1, chunk_size)
for i in range(0, total_rows, chunk_size):
    # Extract the current chunk
    to_unicode = cdli_unicode.iloc[i:i+chunk_size].copy()

    # Your existing code for processing the data goes here

    #1 check sign
    to_unicode['transliteration'] = to_unicode['transliteration'].astype(str).fillna('')

    # Sign to check for existence
    sign_to_check = '……'

    # Method 1: Using the "any" method with a lambda function
    exists = to_unicode.iloc[:, 1].apply(lambda x: sign_to_check in x).any()

    # Method 2: Using the "str.contains" method
    exists = to_unicode.iloc[:, 1].str.contains(sign_to_check).any()

    # Check if the sign exists in the second column
    if exists:
        print(f"The sign '{sign_to_check}' exists in the second column.")
    else:
        print(f"The sign '{sign_to_check}' does not exist in the second column.")

    #2 Replacing spaces with any desired character (e.g., underscore '_')
    to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(' ', '……')

    # get a copy for transliteration format
    to_transliteration = to_unicode.copy()

    to_transliteration['transliteration'] = to_transliteration['transliteration'].str.replace('……', ' ')

    # replace space with separation sign
    to_transliteration['transliteration'] = to_transliteration['transliteration'].str.replace(' ', ' 、 ')

    # Change this to change the range we want to sort on
    to_transliteration.head(1000)

    # Change this to change the range we want to sort on
    to_transliteration = to_transliteration.head(1000)

    # New DataFrame to store unique words
    word_sorting_df = pd.DataFrame(columns=['id_text', 'id_word', 'form'])

    # Iterate through each row in the original DataFrame
    for index, row in to_transliteration.iterrows():
        words = row['transliteration'].split(' 、 ')

        # Filter out words that are not already in word_sorting_df
        unique_words = [word for word in words if word not in word_sorting_df['form'].tolist()]

        # Add unique words to word_sorting_df with the correct id_text and id_word
        word_data = pd.DataFrame({
            'id_text': [row['id_text']] * len(unique_words),
            'id_word': [f"{row['id_text']}.{i}" for i in range(1, len(unique_words) + 1)],
            'form': unique_words
        })

        # Reorder columns: id_word, id_text, form
        word_data = word_data[['id_text', 'id_word', 'form']]

        word_sorting_df = pd.concat([word_sorting_df, word_data[['id_text', 'id_word', 'form']]], ignore_index=True)


    word_sorting_df = word_sorting_df[word_sorting_df['form'] != ',']
    word_sorting_df = word_sorting_df[word_sorting_df['form'] != '[...]']

    # Reset the index of the word_sorting_df
    word_sorting_df.reset_index(drop=True, inplace=True)

    # Sample data for the new columns
    word_sorting_df['clean'] = ''
    word_sorting_df['cuneiform'] = ''
    word_sorting_df['a-Z'] = ''

    # Reset the index and add 1 to make it 1-indexed
    word_sorting_df.index += 1
    word_sorting_df.index.name = 'id_word'

    # Display the DataFrame
    word_sorting_df

    import pandas as pd

    word_sorting_df

    # Define the function to clean the text and filter words
    def clean_column(text):
        # Check if the text contains any special characters
        has_special_chars = any(char in text for char in ['|', '~', '@', '%', '$', '=', '<', '>'])

        if has_special_chars:
            # Step 1: Retain original text
            original_text_AZ = text

            # remove lowercase
            text = ''.join(char for char in text if not char.islower())

            # Step 1: Remove curly brackets (one space added)
            text = text.replace('{', ' ').replace('}', ' ')

            # Step 2: Remove hyphen (one space added)
            text = text.replace('-', ' ')

            # Step 3: Remove specified characters (no space added)
            for char in ['$','#','_','[',']','*','!','?','\\',':',';','^', '|', '~', '@', '%', '$', '...', '=', '<', '>']:
                text = text.replace(char, '')

            # Step 4: Remove period and add one space
            text = text.replace('.', ' ')

            return original_text_AZ, text
        else:
            # Step 1: Remove curly brackets (one space added)
            text = text.replace('{', ' ').replace('}', ' ')

            # Step 2: Remove hyphen (one space added)
            text = text.replace('-', ' ')

            # Step 3: Remove specified characters (no space added)
            for char in ['$','#','_','[',']','*','!','?','\\',':',';','^']:
                text = text.replace(char, '')

            # Step 4: Remove period and add one space
            text = text.replace('.', ' ')

            return text, text

    special_chars = ['|', '~', '@', '%', '$', '...', '=', '<', '>', 'x']

    # Apply the cleaning and filtering function to the 'unique_words' column
    word_sorting_df['a-Z'] = word_sorting_df['form'].apply(lambda text: clean_column(text)[0] if text.isalpha() == False and any(char in text for char in special_chars) == True else "")

    # Create the 'clean' column by applying the same steps and storing the result
    word_sorting_df['clean'] = word_sorting_df['form'].apply(lambda text: clean_column(text)[1] if text != "" else "")

    word_sorting_df

    # 10 convert to sign
    # Create a dictionary from dataframe A mapping signs to unicode
    unicode_dict = dict(zip(concatenated['sign'], concatenated['unicode']))

    def replace_with_unicode(text):
        unicode_values = []
        for sign in text.split():
            unicode = unicode_dict.get(sign)
            if unicode is not None:
                unicode_values.append(unicode)
            else:
                unicode_values.append("N/A")  # Keep the sign if no corresponding unicode value found
        return ' '.join(unicode_values)

    def az_clean_column(text):
        # Step 1: Remove curly brackets (one space added)
        text = text.replace('{', ' ').replace('}', ' ')

        # Step 2: Remove hyphen (one space added)
        text = text.replace('-', ' ')

        # Step 3: Remove specified characters (no space added)
        for char in ['$','#','_','[',']','*','!','?','/','\\',':',';','^']:
            text = text.replace(char, '')

        # Step 4: Remove period and add one space
        text = text.replace('.', ' ')

        return text

    word_sorting_df['cuneiform'] = word_sorting_df['clean'].apply(replace_with_unicode)
    word_sorting_df['cuneiform'] = word_sorting_df['cuneiform'].str.replace(' ', '')

    word_sorting_df

    pattern = r'^[a-zA-Z.,|()#~@%$\'...=<>x]+$'

    # Apply the pattern to the 'a-Z' column and create a new column 'is_valid'
    word_sorting_df['is_valid'] = word_sorting_df['a-Z'].apply(lambda x: bool(re.match(pattern, x)))
    num_true = word_sorting_df['is_valid'].value_counts().get(True, 0)
    print(f"Number in a-Z: {num_true}")

    # Save the processed chunk to CSV with id_word in front of form
    output_filename = f'/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Harrison/alphabetical_{i//chunk_size + 1}.csv'
    word_sorting_df.to_csv(output_filename, index=False)

    # Clear memory
    del to_unicode
    del word_sorting_df

# Finish the program
print("Processing complete.")


Number in a-Z: 62
The sign '……' does not exist in the second column.
Number in a-Z: 91
The sign '……' does not exist in the second column.
Number in a-Z: 101
The sign '……' does not exist in the second column.
Number in a-Z: 78
The sign '……' does not exist in the second column.
Number in a-Z: 36
The sign '……' does not exist in the second column.
Number in a-Z: 42
The sign '……' does not exist in the second column.
Number in a-Z: 44
The sign '……' does not exist in the second column.
Number in a-Z: 78
The sign '……' does not exist in the second column.
Number in a-Z: 506
The sign '……' does not exist in the second column.
Number in a-Z: 30
The sign '……' does not exist in the second column.
Number in a-Z: 53
The sign '……' does not exist in the second column.
Number in a-Z: 32
The sign '……' does not exist in the second column.
Number in a-Z: 47
The sign '……' does not exist in the second column.
Number in a-Z: 49
The sign '……' does not exist in the second column.
Number in a-Z: 1194
The sign '……

KeyboardInterrupt: 

###6.1 Punctuation removal (in each row) with result in 'clean'

See ATF punctuation rules: http://oracc.museum.upenn.edu/doc/help/editinginatf/primer/inlinetutorial/index.html

0. If the word has one of the following, add to a-Z column: `|, ~, @, %, $, '...', =, <, >, x,`
1. Remove curley brackets `{ }` (one space added)
2. Remove hyphen `-` (one space added)
3. Remove: `$, #, _, [, ], *, !, ?, /, \, :, ;, ^ `  (no spaces added)
4. Remove: `.` (one space added)
5. Any numbers followed by parentheses, no changes needed, just move to clean for conversion: `1(disz)`, etc.
6. After moving the words with bars and tildes to a-Z, remove all other punctuation marks in 'clean' (along with any letters immediately following the tilde, see ORACC Alograph list below).

For example, for bars `|` and tilde `~`:
* E.g. `|SZE~a.NAM2|#` ==> clean = `SZE NAM2` | a-Z = `|SZE~a.NAM2|#`
* E.g. `SZITA~a1` ==> clean = `SZITA` | a-Z = `SZITA~a1`


|id_text|id_word|form|clean|a-Z|
|--|--|--|--|--|
|P496727|1|2(disz)|2(disz)||
|P496727|2|me|me||
|P496727|3|udu|udu||
|P496727|4|hi-a|hi-a||
|P496727|5|DUB.SAR|DUB SAR||
|P496727|5|PA~a#?|PA|PA~a#?|
|P496727|5|di@t|di|di@t|

__ORACC Alograph Annotations__

|Modifier|	ATF|	Example	|
|--------|-----|----------|
|curved	|@c	|AŠ@c	|
|flat	|@f	|1(N01@f)	|
|gunu (4 extra wedges)	|@g	|DU@g	|
|sheshig (added še-sign)	|@s	|DU@s	|
|tenu (slanting)	|@t	|GAN₂@t	|
|nutillu (unfinished)	|@n	|SAG@n|
|zidatenu (slanting right)|	@z|	AŠ@z|
|kabatenu (slanting left)	|@k	|AŠ@k|
|vertically reflected	|@r	|U@r|
|horizontally reflected	|@h	|N07~a@h|
|rotated	|@DIGITS	|NAGA@180	|
|variant	|@v	|4(ban₂)@v|

* Adam will review the reulting list of signs that do not get matched to a cuneiform sign. For this we want a CSV of unique tokens (with original punctuation).


In [ ]:
cdli_unicode = pd.read_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/transliteration.csv')
cdli_unicode

,id_text,transliteration
0,P496727,2(disz) me _udu hi-a_ sza su-ga-gu-ut {disz}me...
1,P353473,1(disz) me 1(u) _udu hi-a_ sza su-ga-gu-ut a-a...
2,P496726,i-na 1(disz) ma-na ku3-babbar_ sza su-ga-gu#-u...
3,P496725,i-na 1(disz) me _udu hi-a_ sza su-ga-gu-ut ha-...
4,P496724,i-na _3(disz) 1/3(disz) ma-na ku3-babbar_ 2(di...
...,...,...
132946,P000005,"1(N01) , ESZDA# 1(N01) , KAB 1(N01) , DI@t? 1(..."
132947,P000004,"1(N01) , |SZE~a.NAM2|# 1(N01) , PA~a#? [...] 1..."
132948,P000003,"1(N01) , NI~b# 1(N01) , |NI~bxX| 1(N01) , IR~a..."
132949,P000002,"[1(N01)] , [...] [1(N01)] , GAL~a# SZITA~a1 [1..."


In [ ]:
#this format is for unicode
to_unicode = cdli_unicode.copy()

In [ ]:
#1 check sign

to_unicode['transliteration'] = to_unicode['transliteration'].astype(str).fillna('')


# Sign to check for existence
sign_to_check = '……'

# Method 1: Using the "any" method with a lambda function
exists = to_unicode.iloc[:, 1].apply(lambda x: sign_to_check in x).any()

# Method 2: Using the "str.contains" method
exists = to_unicode.iloc[:, 1].str.contains(sign_to_check).any()

# Check if the sign exists in the second column
if exists:
    print(f"The sign '{sign_to_check}' exists in the second column.")
else:
    print(f"The sign '{sign_to_check}' does not exist in the second column.")


In [ ]:
#2 Replacing spaces with any desired character (e.g., underscore '_')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(' ', '……')

In [ ]:
#get a copy for transliteration format
to_transliteration = to_unicode.copy()

In [ ]:
to_transliteration['transliteration'] = to_transliteration['transliteration'].str.replace('……', ' ')

In [ ]:
#replace space with separation sign
to_transliteration['transliteration'] = to_transliteration['transliteration'].str.replace(' ', ' 、 ')


#Change this to change the range we want to sort on
to_transliteration.head(1000)



,id_text,transliteration
0,P496727,2(disz) 、 me 、 _udu 、 hi-a_ 、 sza 、 su-ga-gu-u...
1,P353473,1(disz) 、 me 、 1(u) 、 _udu 、 hi-a_ 、 sza 、 su-...
2,P496726,i-na 、 1(disz) 、 ma-na 、 ku3-babbar_ 、 sza 、 s...
3,P496725,i-na 、 1(disz) 、 me 、 _udu 、 hi-a_ 、 sza 、 su-...
4,P496724,i-na 、 _3(disz) 、 1/3(disz) 、 ma-na 、 ku3-babb...
...,...,...
995,P203321,2(u) 、 4(asz) 、 3(barig) 、 3(ban2) 、 sze 、 gur...
996,P207675,1(barig) 、 dabin 、 lugal 、 sza3 、 e2-duru5-sze...
997,P203090,1(disz) 、 tug2 、 nig2-lam2 、 lugal 、 lu2-{d}su...
998,P207667,1(barig) 、 dabin 、 lugal 、 ki 、 u4-ne-nig2-sa6...


In [ ]:
#Change this to change the range we want to sort on

to_transliteration = to_transliteration.head(1000)

In [ ]:
# New DataFrame to store unique words
word_sorting_df = pd.DataFrame(columns=['id_text','form'])

# Iterate through each row in the original DataFrame
for index, row in to_transliteration.iterrows():
    words = row['transliteration'].split(' 、 ')

    # Filter out words that are not already in word_sorting_df
    unique_words = [word for word in words if word not in word_sorting_df['form'].tolist()]


    # Add unique words to word_sorting_df
    word_sorting_df = word_sorting_df.append(pd.DataFrame(unique_words, columns=['form']), ignore_index=True)

    word_sorting_df['id_text'] = row['id_text']



word_sorting_df = word_sorting_df[word_sorting_df['form'] != ',']
word_sorting_df = word_sorting_df[word_sorting_df['form'] != '[...]']


# Reset the index of the word_sorting_df
word_sorting_df.reset_index(drop=True, inplace=True)



<ipython-input-50-fba7bac030f0>:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  word_sorting_df = word_sorting_df.append(pd.DataFrame(unique_words, columns=['form']), ignore_index=True)
<ipython-input-50-fba7bac030f0>:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  word_sorting_df = word_sorting_df.append(pd.DataFrame(unique_words, columns=['form']), ignore_index=True)
<ipython-input-50-fba7bac030f0>:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  word_sorting_df = word_sorting_df.append(pd.DataFrame(unique_words, columns=['form']), ignore_index=True)
<ipython-input-50-fba7bac030f0>:13: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.


In [ ]:
word_sorting_df

,id_text,form
0,P203415,2(disz)
1,P203415,me
2,P203415,_udu
3,P203415,hi-a_
4,P203415,sza
...,...,...
9270,P203415,{d}dumu]-zi
9271,P203415,u4-ne-nig2-sa6-ta
9272,P203415,giri3-ni
9273,P203415,nig2-gur11-a-ni


In [ ]:
# Sample data for the new columns
word_sorting_df['clean'] = ''
word_sorting_df['cuneiform'] = ''
word_sorting_df['a-Z'] = ''

# Reset the index and add 1 to make it 1-indexed
word_sorting_df.index += 1
word_sorting_df.index.name = 'id_word'


# Display the DataFrame
word_sorting_df

,id_text,form,clean,cuneiform,a-Z
id_word,,,,,
1,P203415,2(disz),,,
2,P203415,me,,,
3,P203415,_udu,,,
4,P203415,hi-a_,,,
5,P203415,sza,,,
...,...,...,...,...,...
9271,P203415,{d}dumu]-zi,,,
9272,P203415,u4-ne-nig2-sa6-ta,,,
9273,P203415,giri3-ni,,,


In [ ]:
import pandas as pd

word_sorting_df

# Define the function to clean the text and filter words
def clean_column(text):
    # Check if the text contains any special characters
    has_special_chars = any(char in text for char in ['|', '~', '@', '%', '$', '...', '=', '<', '>', 'x'])

    if has_special_chars:
        # Step 1: Retain original text
        original_text_AZ = text

        # remove lowercase
        text = ''.join(char for char in text if not char.islower())

        # Step 1: Remove curly brackets (one space added)
        text = text.replace('{', ' ').replace('}', ' ')

        # Step 2: Remove hyphen (one space added)
        text = text.replace('-', ' ')

        # Step 3: Remove specified characters (no space added)
        for char in ['$','#','_','[',']','*','!','?','/','\\',':',';','^', '|', '~', '@', '%', '$', '...', '=', '<', '>', 'x']:
            text = text.replace(char, '')

        # Step 4: Remove period and add one space
        text = text.replace('.', ' ')

        return original_text_AZ, text
    else:
        # Step 1: Remove curly brackets (one space added)
        text = text.replace('{', ' ').replace('}', ' ')

        # Step 2: Remove hyphen (one space added)
        text = text.replace('-', ' ')

        # Step 3: Remove specified characters (no space added)
        for char in ['$','#','_','[',']','*','!','?','/','\\',':',';','^']:
            text = text.replace(char, '')

        # Step 4: Remove period and add one space
        text = text.replace('.', ' ')

        return text, text

special_chars = ['|', '~', '@', '%', '$', '...', '=', '<', '>', 'x']

# Apply the cleaning and filtering function to the 'unique_words' column
word_sorting_df['a-Z'] = word_sorting_df['form'].apply(lambda text: clean_column(text)[0] if text.isalpha() == False and any(char in text for char in special_chars) == True else "")

# Create the 'clean' column by applying the same steps and storing the result
word_sorting_df['clean'] = word_sorting_df['form'].apply(lambda text: clean_column(text)[1] if text != "" else "")

word_sorting_df

,id_text,id_word,form,a-Z,clean
0,P262543,P262543.1,1(disz),,1(disz)
1,P262543,P262543.2,udu,,udu
2,P262543,P262543.3,nidba,,nidba
3,P262543,P262543.4,e2,,e2
4,P262543,P262543.5,{d}en-ki,,d en ki
...,...,...,...,...,...
3635,P285270,P285270.24,szu-pu-u,,szu pu u
3636,P285270,P285270.25,"{m}ki-s,ir-_{d}muati",,"m ki s,ir d muati"
3637,P429644,P429644.1,nig2]-bar#-ra,,nig2 bar ra
3638,P429644,P429644.2,sze#,,sze


###6.2 Match 'clean' sign readings to unicode cuneiform
Using the combined dictionary the two sign lists, match the sign readings on each row to unicode equivalents.

|id_text|id_word|form|clean |cuneiform|a-Z|
|--|--|--|--|--|--|
|P496727|1|2(disz)|2(disz)|𒈫||
|P496727|2|me|me|𒈨||
|P496727|3|udu|udu|𒇻||
|P496727|4|hi-a|hi a|𒄭𒀀||

In [ ]:
#10 convert to sign
# Create a dictionary from dataframe A mapping signs to unicode
unicode_dict = dict(zip(concatenated['sign'], concatenated['unicode']))

def replace_with_unicode(text):
    unicode_values = []
    for sign in text.split():
        unicode = unicode_dict.get(sign)
        if unicode is not None:
            unicode_values.append(unicode)
        else:
            unicode_values.append("N/A")  # Keep the sign if no corresponding unicode value found
    return ' '.join(unicode_values)

word_sorting_df['cuneiform'] = word_sorting_df['clean'].apply(replace_with_unicode)
word_sorting_df['cuneiform'] = word_sorting_df['cuneiform'].str.replace(' ', '')

word_sorting_df


,id_text,id_word,form,a-Z,clean,cuneiform
0,P262543,P262543.1,1(disz),,1(disz),𒁹
1,P262543,P262543.2,udu,,udu,𒇻
2,P262543,P262543.3,nidba,,nidba,𒉻𒀭𒈹
3,P262543,P262543.4,e2,,e2,𒂍
4,P262543,P262543.5,{d}en-ki,,d en ki,𒀭𒂗𒆠
...,...,...,...,...,...,...
3635,P285270,P285270.24,szu-pu-u,,szu pu u,𒋗𒁍𒌋
3636,P285270,P285270.25,"{m}ki-s,ir-_{d}muati",,"m ki s,ir d muati",𒁹𒆠𒈲𒀭𒉺
3637,P429644,P429644.1,nig2]-bar#-ra,,nig2 bar ra,𒃻𒁇𒊏
3638,P429644,P429644.2,sze#,,sze,𒊺


###6.3 Assessment of unicode conversion

1. Check to see if the 'a-Z' column contains the correct data: alphabet characters and any remaining special characters:  ., |, (, ), #, ~, @, %, $, '...', =, <, >, x
2. Export the resulting data frame to CSV
3. Iterative Save to CSV

* Save to CSV after 100,000 rows
* File: CDLI_signs_001.csv
* Folder: Save to Harrison/CDLI_signs


|id_text|id_word|form|clean|cuneiform|a-Z|
|--|--|--|--|--|--|
|P496727|1|2(disz)|2(disz)|𒈫||
|P496727|2|me|me|𒈨||
|P496727|3|udu|udu|𒇻||
|P496727|4|hi-a|hi a|𒄭𒀀|||
|P000005|1|1(n01)|||1(n01)|

In [ ]:
import re
pattern = r'^[a-zA-Z.,|()#~@%$\'...=<>x]+$'

# Apply the pattern to the 'a-Z' column and create a new column 'is_valid'
word_sorting_df['is_valid'] = word_sorting_df['a-Z'].apply(lambda x: bool(re.match(pattern, x)))
num_true = word_sorting_df['is_valid'].value_counts().get(True, 0)
print(f"Number in a-Z: {num_true}")

Number in a-Z: 41


In [ ]:
word_sorting_df.to_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Harrison/alphabetical.csv')

###6.4 Evaluation for Resulting Unmatched Sign Readings

With the assessment in section 6.3, we want to make a list of signs which have failed to find their equivalent Unicode signs. This list can be used to prepare a unicode proposal and track updates in this conversion pipeline.

1. Read in the alphabetical.csv files (probably best done iteratively): https://drive.google.com/drive/folders/1mi23tcw_2-_mvZ5rgTB3mKed_Ea9Q85F?usp=drive_link
2. Filter 'cuneiform' column for N/A values or blank values
3. Make a new data frame with the filtered results (i.e. all the columns for a row with N/A)
4. Make a second data frame of unique 'form' values, and include a count for how many times these N/A forms occurred.
5. Export the second data frame of unique forms, their counts, and the list of text IDs where these forms occur.

__Example for CSV Export:__

|form_unique|forms|cuneiform|count|id_text|id_word|
|--|--|--|--|--|--|
|single string|multiple strings in a list|multiple strings in a list|integer|list of IDs|list of IDs
|{gesz}giparx(KISAL)|gesz giparx(KISAL)|𒄑N/A|1|P207586|P207586.1|
|tibirx(KU)|'ur-bad3-tibirx(KU)-ra-ta', 'ur-bad3-tibirx(KU)-ra' 'ur-bad3-tibirx(KU)-ra-sze3'|'𒌨𒂦N/A𒊏𒋫', '𒌨𒂦N/A𒊏', '𒌨𒂦N/A𒊏𒂠'|3|'P206038', 'P203975', 'P205735'|'P206038.1', 'P203975.3', 'P205735.2'|

###6.5 Filter results

In this section we will remove all the valid sign readings which can be matched to the signlist when punctuation is removed.

* For unique_forms_df, if we remove the punctuation, can we find a match?
  * yes, then filter from the data fram
  * no, then keep it in the list



In [ ]:
def clean_column(text):
        # Check if the text contains any special characters
        has_special_chars = any(char in text for char in ['|', '~', '@', '%', '$', '=', '<', '>'])

        if has_special_chars:
            # Step 1: Retain original text
            original_text_AZ = text

            # remove lowercase
            text = ''.join(char for char in text if not char.islower())

            # Step 1: Remove curly brackets (one space added)
            text = text.replace('{', ' ').replace('}', ' ')

            # Step 2: Remove hyphen (one space added)
            text = text.replace('-', ' ')

            # Step 3: Remove specified characters (no space added)
            for char in ['$','#','_','[',']','*','!','?','\\',':',';','^', '|', '~', '@', '%', '$', '...', '=', '<', '>']:
                text = text.replace(char, '')

            # Step 4: Remove period and add one space
            text = text.replace('.', ' ')

            return original_text_AZ, text
        else:
            # Step 1: Remove curly brackets (one space added)
            text = text.replace('{', ' ').replace('}', ' ')

            # Step 2: Remove hyphen (one space added)
            text = text.replace('-', ' ')

            # Step 3: Remove specified characters (no space added)
            for char in ['$','#','_','[',']','*','!','?','\\',':',';','^']:
                text = text.replace(char, '')

            # Step 4: Remove period and add one space
            text = text.replace('.', ' ')

            return text, text


unicode_dict = dict(zip(concatenated['sign'], concatenated['unicode']))

def replace_with_unicode(text):
    unicode_values = []
    for sign in text.split():
        unicode = unicode_dict.get(sign)
        if unicode is not None:
            unicode_values.append(unicode)
        else:
            unicode_values.append("N/A")  # Keep the sign if no corresponding unicode value found
    return ' '.join(unicode_values)

In [ ]:
import os
import pandas as pd

# Define the directory where CSV files are saved
csv_directory = '/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Harrison/'


# Initialize an empty list to store filtered DataFrames
filtered_dfs = []

# Iterate over each CSV file in the directory
for file_name in os.listdir(csv_directory):
    if file_name.endswith('.csv') and 'alphabetical' in file_name:
        file_path = os.path.join(csv_directory, file_name)
        # Read the CSV file
        df = pd.read_csv(file_path)
        # Filter 'cuneiform' column for N/Aa values or blank values
        filtered_rows = df[df['cuneiform'].isna() | df['cuneiform'].eq('')]
        filtered_rows['form'] = filtered_rows['form'].astype(str)
        # Append filtered rows to the list
        filtered_dfs.append(filtered_rows)

# Concatenate all filtered DataFrames into one DataFrame
filtered_df = pd.concat(filtered_dfs)


#['|', '~', '@', '%', '$', '...', '=', '<', '>', 'x']


filtered_df['form'] = filtered_df['form'].apply(lambda x: x.lower() if isinstance(x, str) else x)
filtered_df['form'] = filtered_df['form'].str.replace('<', '')
filtered_df['form'] = filtered_df['form'].str.replace('>', '')
filtered_df['form'] = filtered_df['form'].str.replace('?', '')
filtered_df['form'] = filtered_df['form'].str.replace(']', '')
filtered_df['form'] = filtered_df['form'].str.replace('[', '')
filtered_df['form'] = filtered_df['form'].str.replace('_', '')


filtered_df['clean'] = filtered_df['form'].apply(lambda text: clean_column(text)[1] if text != "" else "")
filtered_df['cuneiform'] = filtered_df['clean'].apply(replace_with_unicode)
filtered_df['cuneiform'] = filtered_df['cuneiform'].str.replace(' ', '')



filtered_df



<ipython-input-71-2b5de192b1b4>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_rows['form'] = filtered_rows['form'].astype(str)
<ipython-input-71-2b5de192b1b4>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_rows['form'] = filtered_rows['form'].astype(str)
<ipython-input-71-2b5de192b1b4>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://p

,id_word,id_text,form,clean,cuneiform,a-Z,is_valid
48,49,P203415,1/2(disz),1/2(disz),𒈦,NaN,False
79,80,P203415,2/3(disz),2/3(disz),𒑛,NaN,False
90,91,P203415,...,,,...],False
108,109,P203415,1/3(disz),1/3(disz),𒑚,NaN,False
116,117,P203415,1/3(disz),1/3(disz),𒑚,NaN,False
...,...,...,...,...,...,...,...
18382,P220655.95,P220655,gan2,gan2,𒃷,NaN,False
18384,P220655.97,P220655,gan2,gan2,𒃷,NaN,False
18388,P220655.101,P220655,3(u@c),3(),N/A,_3(u@c),False
18399,P220655.112,P220655,2(u@c),2(),N/A,_2(u@c),False


In [ ]:
count = filtered_df['form'].str.count("<").sum()
count

0

__Example for CSV Export:__

|form_unique|forms|cuneiform|count|id_text|id_word|
|--|--|--|--|--|--|
|single string|multiple strings in a list|multiple strings in a list|integer|list of IDs|list of IDs
|{gesz}giparx(KISAL)|gesz giparx(KISAL)|𒄑N/A|1|P207586|P207586.1|
|tibirx(KU)|'ur-bad3-tibirx(KU)-ra-ta', 'ur-bad3-tibirx(KU)-ra' 'ur-bad3-tibirx(KU)-ra-sze3'|'𒌨𒂦N/A𒊏𒋫', '𒌨𒂦N/A𒊏', '𒌨𒂦N/A𒊏𒂠'|3|'P206038', 'P203975', 'P205735'|'P206038.1', 'P203975.3', 'P205735.2'|

In [ ]:
filtered_df = filtered_df[filtered_df['cuneiform']=='N/A']
filtered_df

,id_word,id_text,form,clean,cuneiform,a-Z,is_valid
127,128,P203415,($,(,N/A,[($,False
128,129,P203415,blank,blank,N/A,NaN,False
129,130,P203415,space,space,N/A,NaN,False
130,131,P203415,$),),N/A,$),True
147,148,P203415,($,(,N/A,($,True
...,...,...,...,...,...,...,...
18376,P220655.89,P220655,2(asz@c),2(),N/A,_2(asz@c),False
18378,P220655.91,P220655,2(asz@c),2(),N/A,_2(asz@c),False
18379,P220655.92,P220655,menx(|ga2xen|)-mu,(2),N/A,menx(|GA2xEN|)-mu,False
18388,P220655.101,P220655,3(u@c),3(),N/A,_3(u@c),False


In [ ]:
# Create a DataFrame of unique 'form' values along with their counts and text IDs
unique_forms_df = filtered_df.groupby('clean').agg({'form': lambda x: list(x), 'cuneiform': lambda x: list(x), 'id_text': lambda x: list(x), 'id_word': lambda x: list(x)}).reset_index()

def count_elements_in_list(lst):
    return len(lst)

# Apply the function to each list column and store the result in a new column 'count'
unique_forms_df['count'] = unique_forms_df['form'].apply(count_elements_in_list)


unique_forms_df.columns = ['form_unique', 'forms', 'cuneiform', 'id_text', 'id_word','count']

unique_forms_df


,form_unique,forms,cuneiform,id_text,id_word,count
0,(,[{d}na-bi-um-na-($],[N/A],[P518978],[P518978.1],1
1,",","[a-r-t-x-$-a-c,-a]",[N/A],[P522162],[P522162.16],1
2,5(),[{d}nin-tin{ti}-ug5(|ezemxan|)-ga],[N/A],[P010570],[P010570.20],1
3,(,[sa-bu-um{ki}-($],[N/A],[P380571],[P380571.28],1
4,(),[{d}suen-sza-ru-uh!(|hixdisz|)],[N/A],[P224335],[P224335.11],1
...,...,...,...,...,...,...
4465,���lugal,[���lugal#],[N/A],[P137890],[P137890.6],1
4466,���na���,[���na���],[N/A],[P498964],[P498964.9],1
4467,���ul,[���ul#],[N/A],[P238860],[P238860.17],1
4468,���x,"[���x, ���x, ���x, ���x, ���x, ���x, ���x]","[N/A, N/A, N/A, N/A, N/A, N/A, N/A]","[P394325, P394325, P394325, P394325, P396882, ...","[P394325.47, P394325.129, P394325.309, P394325...",7


In [ ]:
def extract_unique_words(row):
    return list(set(row))

# Apply the function to each row in the 'form' column
unique_forms_df['form_unique'] = unique_forms_df['forms'].apply(extract_unique_words)

unique_forms_df['count_unique'] = unique_forms_df['form_unique'].apply(count_elements_in_list)


In [ ]:
unique_forms_df = unique_forms_df.sort_values('count', ascending = False)
unique_forms_df

,form_unique,forms,cuneiform,id_text,id_word,count,count_unique
609,"[1(disz@t), 1(u@c)#*, 1(asz@v)!, uru~a1!(gurus...","[1(asz@c), 1(asz@c), 1(asz@c), 1(barig@c), 1(b...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ...","[P203415, P203415, P203415, P203415, P203415, ...","[5673, 5678, 5683, 5829, 5842, 5843, 5848, 585...",3547,56
1093,"[2(asz@t), murub2(|munusxlagar|), 2(asz@c)@, 2...","[2(iku@c), 2(iku@c), 2(iku@c), 2(iku@c), 2(bar...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ...","[P203415, P203415, P203415, P203415, P203415, ...","[5646, 5655, 5657, 5664, 5690, 5763, 5764, 576...",1680,64
1465,"[3(barig@c)*, 3(disz@t)*, 3(disz@t)!, |nax3(as...","[3(iku@c), 3(iku@c), 3(iku@c), 3(iku@c), 3(u@c...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ...","[P203415, P203415, P203415, P203415, P203415, ...","[5645, 5660, 5662, 5667, 5688, 5689, 5801, 915...",1227,43
1798,"[4(u@c)#, 4(gesz@c), 4(disz@t):iti, 4(barig@c)...","[4(iku@c), 4(u@c), 4(asz@c), 4(disz@c), 4(u@c)...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ...","[P203415, P203415, P203415, P203415, P416079, ...","[5671, 5769, 5956, 9155, P416079.1, P275455.15...",893,41
173,"[|sze&sze|, |sze~a&sze~a|, |zi&zi|, |sze~a&sze...","[&, &, &, &, &, &, &, &, &, &, &, &, &, &, &, ...","[N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, N/A, ...","[P203415, P203415, P203415, P203415, P203415, ...","[5240, 5241, 5243, 5246, 5248, 5250, 5251, 525...",772,43
...,...,...,...,...,...,...,...
2601,[di/ki],[di/ki],[N/A],[P258174],[P258174.10],1,1
2602,[dida!(sa)],[dida!(sa)],[N/A],[P378821],[P378821.2],1,1
661,[1(asz45)],[1(asz45)],[N/A],[P213040],[P213040.4],1,1
1966,[490],[490],[N/A],[P229061],[P229061.44],1,1


In [ ]:
unique_forms_df.to_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Harrison/unique_forms.csv', index=False)

In [ ]:
sum((filtered_df['cuneiform'] == "N/A").values.astype("int"))

40172

##7 ORACC dataset for conversion

* [finaldf.csv](https://drive.google.com/file/d/1_Jp4OVDy9et0G-Wf5cnL8VMk9NAAcmyB/view?usp=drive_link)

1. See if we can convert the ORACC text to clean and cuneiform using the same rules.

In [ ]:
#ORACC version

import pandas as pd
import re

# Read the CSV file
cdli_unicode = pd.read_csv('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/databases/CDLI/transliteration.csv')

# Process data in segments of 1000 rows
chunk_size = 1000
total_rows = len(cdli_unicode)

# number of rows can change for shorter runs, e.g. (0, total_rows, chunk_size) > (0, 1, chunk_size)
for i in range(0, total_rows, chunk_size):
    # Extract the current chunk
    to_unicode = cdli_unicode.iloc[i:i+chunk_size].copy()

    # Your existing code for processing the data goes here

    #1 check sign
    to_unicode['transliteration'] = to_unicode['transliteration'].astype(str).fillna('')

    # Sign to check for existence
    sign_to_check = '……'

    # Method 1: Using the "any" method with a lambda function
    exists = to_unicode.iloc[:, 1].apply(lambda x: sign_to_check in x).any()

    # Method 2: Using the "str.contains" method
    exists = to_unicode.iloc[:, 1].str.contains(sign_to_check).any()

    # Check if the sign exists in the second column
    if exists:
        print(f"The sign '{sign_to_check}' exists in the second column.")
    else:
        print(f"The sign '{sign_to_check}' does not exist in the second column.")

    #2 Replacing spaces with any desired character (e.g., underscore '_')
    to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(' ', '……')

    # get a copy for transliteration format
    to_transliteration = to_unicode.copy()

    to_transliteration['transliteration'] = to_transliteration['transliteration'].str.replace('……', ' ')

    # replace space with separation sign
    to_transliteration['transliteration'] = to_transliteration['transliteration'].str.replace(' ', ' 、 ')

    # Change this to change the range we want to sort on
    to_transliteration.head(1000)

    # Change this to change the range we want to sort on
    to_transliteration = to_transliteration.head(1000)

    # New DataFrame to store unique words
    word_sorting_df = pd.DataFrame(columns=['id_text', 'id_word', 'form'])

    # Iterate through each row in the original DataFrame
    for index, row in to_transliteration.iterrows():
        words = row['transliteration'].split(' 、 ')

        # Filter out words that are not already in word_sorting_df
        unique_words = [word for word in words if word not in word_sorting_df['form'].tolist()]

        # Add unique words to word_sorting_df with the correct id_text and id_word
        word_data = pd.DataFrame({
            'id_text': [row['id_text']] * len(unique_words),
            'id_word': [f"{row['id_text']}.{i}" for i in range(1, len(unique_words) + 1)],
            'form': unique_words
        })

        # Reorder columns: id_word, id_text, form
        word_data = word_data[['id_text', 'id_word', 'form']]

        word_sorting_df = word_sorting_df.append(word_data, ignore_index=True)

    word_sorting_df = word_sorting_df[word_sorting_df['form'] != ',']
    word_sorting_df = word_sorting_df[word_sorting_df['form'] != '[...]']

    # Reset the index of the word_sorting_df
    word_sorting_df.reset_index(drop=True, inplace=True)

    # Sample data for the new columns
    word_sorting_df['clean'] = ''
    word_sorting_df['cuneiform'] = ''
    word_sorting_df['a-Z'] = ''

    # Reset the index and add 1 to make it 1-indexed
    word_sorting_df.index += 1
    word_sorting_df.index.name = 'id_word'

    # Display the DataFrame
    word_sorting_df

    import pandas as pd

    word_sorting_df

    # Define the function to clean the text and filter words
    def clean_column(text):
        # Check if the text contains any special characters
        has_special_chars = any(char in text for char in ['|', '~', '@', '%', '$', '=', '<', '>'])

        if has_special_chars:
            # Step 1: Retain original text
            original_text_AZ = text

            # remove lowercase
            text = ''.join(char for char in text if not char.islower())

            # Step 1: Remove curly brackets (one space added)
            text = text.replace('{', ' ').replace('}', ' ')

            # Step 2: Remove hyphen (one space added)
            text = text.replace('-', ' ')

            # Step 3: Remove specified characters (no space added)
            for char in ['$','#','_','[',']','*','!','?','\\',':',';','^', '|', '~', '@', '%', '$', '...', '=', '<', '>']:
                text = text.replace(char, '')

            # Step 4: Remove period and add one space
            text = text.replace('.', ' ')

            return original_text_AZ, text
        else:
            # Step 1: Remove curly brackets (one space added)
            text = text.replace('{', ' ').replace('}', ' ')

            # Step 2: Remove hyphen (one space added)
            text = text.replace('-', ' ')

            # Step 3: Remove specified characters (no space added)
            for char in ['$','#','_','[',']','*','!','?','\\',':',';','^']:
                text = text.replace(char, '')

            # Step 4: Remove period and add one space
            text = text.replace('.', ' ')

            return text, text

    special_chars = ['|', '~', '@', '%', '$', '...', '=', '<', '>', 'x']

    # Apply the cleaning and filtering function to the 'unique_words' column
    word_sorting_df['a-Z'] = word_sorting_df['form'].apply(lambda text: clean_column(text)[0] if text.isalpha() == False and any(char in text for char in special_chars) == True else "")

    # Create the 'clean' column by applying the same steps and storing the result
    word_sorting_df['clean'] = word_sorting_df['form'].apply(lambda text: clean_column(text)[1] if text != "" else "")

    word_sorting_df

    # 10 convert to sign
    # Create a dictionary from dataframe A mapping signs to unicode
    unicode_dict = dict(zip(concatenated['sign'], concatenated['unicode']))

    def replace_with_unicode(text):
        unicode_values = []
        for sign in text.split():
            unicode = unicode_dict.get(sign)
            if unicode is not None:
                unicode_values.append(unicode)
            else:
                unicode_values.append("N/A")  # Keep the sign if no corresponding unicode value found
        return ' '.join(unicode_values)

    def az_clean_column(text):
        # Step 1: Remove curly brackets (one space added)
        text = text.replace('{', ' ').replace('}', ' ')

        # Step 2: Remove hyphen (one space added)
        text = text.replace('-', ' ')

        # Step 3: Remove specified characters (no space added)
        for char in ['$','#','_','[',']','*','!','?','/','\\',':',';','^']:
            text = text.replace(char, '')

        # Step 4: Remove period and add one space
        text = text.replace('.', ' ')

        return text

    word_sorting_df['cuneiform'] = word_sorting_df['clean'].apply(replace_with_unicode)
    word_sorting_df['cuneiform'] = word_sorting_df['cuneiform'].str.replace(' ', '')

    word_sorting_df

    pattern = r'^[a-zA-Z.,|()#~@%$\'...=<>x]+$'

    # Apply the pattern to the 'a-Z' column and create a new column 'is_valid'
    word_sorting_df['is_valid'] = word_sorting_df['a-Z'].apply(lambda x: bool(re.match(pattern, x)))
    num_true = word_sorting_df['is_valid'].value_counts().get(True, 0)
    print(f"Number in a-Z: {num_true}")

    # Save the processed chunk to CSV with id_word in front of form
    output_filename = f'/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Harrison/alphabetical_{i//chunk_size + 1}.csv'
    word_sorting_df.to_csv(output_filename, index=False)

    # Clear memory
    del to_unicode
    del word_sorting_df

# Finish the program
print("Processing complete.")


##8 Include Leiden Transcription readings in Updated Sign List Dictionary

We will want to convert text like this to unicode cuneiform as well:

https://docs.google.com/spreadsheets/d/11AedPBoSWWPQbjXKel9nZ_br-Dgs0LyiGm6ehKtsEvY/edit?usp=sharing

Here are some of the changes we could apply:

| CDLI | Print |
| ---- | ----- |
| a2 | á |
| a3 | à |
| e2 | é |
| e3 | è |
| i2 | í |
| i3 | ì |
| u2 | ú |
| u3 | ù |
|h | ḫ|
|ŋ|ĝ|

### Conversions for consonant-vowel-consonant-number (CVC#): for example
* šam2 = šám
* gan2 = gán
* nam2 = nám
* muš3 = mùš
* tur3 = tùr
* gub3 = gùb

### Recommended Order of Operations when converting:

* Basic principle: convert consonant (C) + vowel (V) + number (#) to vowel (V) + accent (see above):

1. If the number is > 100, ignore / keep number
2. If the sign is CVCV# & VCV#, ignore / keep number
3. CVC# convert first (hardest)
4. VC# convert second (next hardest)
5. CV# convert last (easiest)


* Here are the notebooks that we worked on previously to address these changes:
https://colab.research.google.com/drive/1y32iMWhnUapRsTMYDlcQQ6jmh7AKhy0N?usp=drive_link
* Developing Rules to Parse CDLI format:
https://docs.google.com/document/d/1aEfuW0DOqH2R6eM-zPI8_Now73keNgw4RHKTvt-uEm4/edit#heading=h.dklvesms3eba
* Using code from: https://colab.research.google.com/drive/1y32iMWhnUapRsTMYDlcQQ6jmh7AKhy0N?usp=drive_link

In [ ]:
enhance_sign_list = concatenated.copy()

enhance_sign_list = enhance_sign_list.dropna()

concatenated = enhance_sign_list.dropna()

enhance_sign_list.drop(enhance_sign_list.columns[-1], axis=1, inplace=True)

enhance_sign_list

,sign,unicode
0,...aš,𒈏
1,...ba,𒆠𒇴
2,...bun,𒇮
3,...e,𒃬
4,...ga,𒆠𒁉𒆕
...,...,...
16647,6(bur3),𒐑
16648,7(bur3),𒐒
16649,8(bur3),𒐓
16650,9(bur3),𒐔


In [ ]:
text = list(enhance_sign_list["sign"])

In [ ]:
def s_to_lst(s):
  s = s.strip().lower()
  return s.split()

In [ ]:
def remove_underscore(word):
  """
  we do not have enough data to train a cased LM,
  so we remove all cap indicator in CDLI data,
  """
  word = word.replace("_", "")
  return word

In [ ]:
def change_sz(word):
  """
  Chaneg sz into š
  """
  return word.replace("sz", "š")

In [ ]:
def mark_superscript(word):
  """
  In CDLI, superscript is marked in {},
  for training, we want to have a special token for it,
  let's make a SUPER_S, SUPER_E for { and }
  """
  word = word.replace("{", " [SUPER_S] ")
  word = word.replace("}", " [SUPER_E]")
  return word

In [ ]:
two_dict = {"a": "á", "e":"é", "i":"í", "u": "ú"}
three_dict = {"a": "à", "e":"è", "i":"ì", "u": "ù"}

In [ ]:
def process_number(word):
  """
  process number in word,
  change number into sign
  """
  if "2" in word and "2" != word[0]:
    num_index = word.index("2")
    temp = list(word)
    if temp[num_index-1] in ["a", "e", "i", "u"]:
      temp[num_index-1] = two_dict[temp[num_index-1]]
      word = "".join(temp)
      word = word.replace("2", "")
    else:
      word = "".join(temp)
  if "3" in word and "3" != word[0]:
    num_index = word.index("3")
    temp = list(word)
    if temp[num_index-1] in ["a", "e", "i", "u"]:
      temp[num_index-1] = three_dict[temp[num_index-1]]
      word = "".join(temp)
      word = word.replace("3", "")
    else:
      word = "".join(temp)
  return word

In [ ]:
def remove_hashtag(word):
  word = word.replace("#", "")
  return word

In [ ]:
def remove_uncertain(word):
  word = word.replace("[", "")
  word = word.replace("]", "")
  return word

In [ ]:
def parse_sentence(s):
  words = s_to_lst(s)
  parsed = []
  for word in words:
    if word == "[...]":
      continue
    word = remove_underscore(word)
    word = change_sz(word)
    word = mark_superscript(word)
    word = process_number(word)
    word = remove_hashtag(word)
    parsed.append(word)
  return ' '.join(parsed)

In [ ]:
processed = []
for line in progress_bar(text):
  processed.append(parse_sentence(line))
enhance_sign_list["parsed"] = processed

In [ ]:
processed[0]

'...aš'

In [ ]:
enhance_sign_list

,sign,unicode,parsed
0,...aš,𒈏,...aš
1,...ba,𒆠𒇴,...ba
2,...bun,𒇮,...bun
3,...e,𒃬,...e
4,...ga,𒆠𒁉𒆕,...ga
...,...,...,...
16647,6(bur3),𒐑,6(bur3)
16648,7(bur3),𒐒,7(bur3)
16649,8(bur3),𒐓,8(bur3)
16650,9(bur3),𒐔,9(bur3)


In [ ]:
# enhance_sign_list = enhance_sign_list.iloc[:, [2, 1, 0] + list(range(3, enhance_sign_list.shape[1]))]
# enhance_sign_list.drop(enhance_sign_list.columns[-1], axis=1, inplace=True)
# enhance_sign_list.rename(columns={enhance_sign_list.columns[0]: 'sign'}, inplace=True)


# enhance_sign_list

In [ ]:
concatenated

,sign,unicode,_merge
0,...aš,𒈏,sign_list
1,...ba,𒆠𒇴,both
2,...bun,𒇮,both
3,...e,𒃬,both
4,...ga,𒆠𒁉𒆕,both
...,...,...,...
16647,6(bur3),𒐑,akkademia
16648,7(bur3),𒐒,akkademia
16649,8(bur3),𒐓,akkademia
16650,9(bur3),𒐔,akkademia


In [ ]:
common_columns = list(set(concatenated.columns) & set(enhance_sign_list.columns))

missing_rows = enhance_sign_list[~enhance_sign_list['parsed'].isin(concatenated['sign'])]

missing_rows_df = pd.DataFrame(columns=concatenated.columns)

missing_rows_df = missing_rows_df.append(missing_rows)

missing_rows_df

<ipython-input-91-4e304f43bafe>:7: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  missing_rows_df = missing_rows_df.append(missing_rows)


,sign,unicode,_merge,parsed
4461,kwu318,𒆶,NaN,kwù18
8398,{+ab₂}abrig,𒉣𒈨𒁺,NaN,[SUPER_S] +ab₂ [SUPER_E]abrig
8399,{kaš}dida,𒁉𒌑𒊓,NaN,[SUPER_S] kaš [SUPER_E]dida
8400,{kaš}dida₂,𒁉𒍑𒊓,NaN,[SUPER_S] kaš [SUPER_E]dida₂
8401,{kaš}ulušin,𒁉𒀾𒀀𒀭,NaN,[SUPER_S] kaš [SUPER_E]ulušin
...,...,...,...,...
16636,5(gesz'u@c),𒐢,NaN,5(geš'u@c)
16637,limmu3,𒐏,NaN,limmù
16638,limu3,𒐏,NaN,limù
16641,szanabaku2,𒐏,NaN,šanabakú


## CSV Export

This is the list of signs that have a difficult sign reading, and therefore might need checking.